In [3]:
pip install gymnasium

Note: you may need to restart the kernel to use updated packages.


In [4]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

myspace = spaces.Box(low = -2, high = 2, shape = (2,3,6,4), dtype=np.float32)
myspace.sample()

array([[[[ 1.7939385 ,  0.21220241, -0.99668115, -0.4009616 ],
         [-0.09381162, -0.08255197, -0.23706287,  1.3896499 ],
         [ 0.36694458, -1.2201353 ,  0.5432728 , -0.9475129 ],
         [-0.60960376, -1.8179137 , -1.6929078 , -0.5836721 ],
         [-1.6611531 ,  1.8025653 ,  1.5700334 , -0.19985388],
         [ 1.1948217 , -0.65666616,  1.1184348 ,  1.8240436 ]],

        [[ 1.1718596 ,  0.16659029,  1.4686675 ,  1.904507  ],
         [-1.3677427 ,  0.35246107,  0.7818692 , -0.07223995],
         [-1.5394348 ,  1.5575204 , -0.06373931, -1.452378  ],
         [-1.6152053 , -0.22770098,  1.3699359 , -0.49564865],
         [-1.7941703 ,  1.2518665 ,  0.9364785 , -1.6170585 ],
         [-1.3727556 ,  0.6413462 , -0.15347357,  0.89291614]],

        [[-1.185956  ,  0.38703597,  0.57513934, -1.8016374 ],
         [ 0.54457605, -0.21221748, -0.5046676 ,  0.7607698 ],
         [-1.1820688 , -0.36632168, -0.5027234 ,  0.9119886 ],
         [ 1.870027  , -1.1376891 , -0.9430462 , -1

In [5]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

myspace = spaces.MultiDiscrete([4]* 5)

action_space = spaces.Tuple(spaces.Discrete(4) for _ in range(5))

print(myspace.sample())
print(action_space.sample())

[0 3 2 0 3]
(np.int64(0), np.int64(3), np.int64(1), np.int64(1), np.int64(2))


In [6]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

myspace = spaces.MultiBinary(5)

action_space = spaces.MultiBinary((5, 7))

print(myspace.sample())
print(action_space.sample())


[1 0 1 0 0]
[[0 1 0 0 1 1 1]
 [0 1 1 1 0 1 0]
 [0 1 0 1 1 0 0]
 [0 1 1 0 0 1 0]
 [1 1 1 0 1 1 0]]


In [7]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

myspace = spaces.Discrete(7)

action_space = spaces.Box(0.0, float(7), shape=(1,), dtype=np.float32)

print(myspace.sample())
print(action_space.sample())

2
[6.143289]


In [8]:
# Observation space
observation_space = spaces.Tuple((
   spaces.Box(-np.inf, np.inf, shape=(G,), dtype=np.float32),
   spaces.Box(
   -np.inf,
   np.inf,
   shape=(K_MAX, N_OBS, 5),
    dtype=np.float32,),
   spaces.MultiBinary(K_MAX),
   spaces.MultiBinary((K_MAX, N_OBS)),
   spaces.Box(0.0, float(N_DAYS), shape=(1,), dtype=np.float32),
        ))

# Action space: per-candidate {0,1,2,3}
action_space = spaces.MultiDiscrete([4] * K_MAX)

NameError: name 'G' is not defined

In [2]:
#!/usr/bin/env python3
"""
kn_sedm_followup_env_scheduler_no_cadence.py

Full Gymnasium environment for kilonova follow-up with:
- variable number of candidates
- masked fixed-size observation tensor
- nightly PENDING queue
- scheduler inspired by the SEDM logic
- NO historical SEDM cadence
- exactly one executed SEDM batch per night in g, r, i
- reward from improvement in chi^2 of a linear MAGNITUDE-decay model

Important conventions
---------------------
1) Action meanings:
   0 = no request
   1 = 90 s batch total for g+r+i
   2 = 120 s batch total for g+r+i
   3 = 180 s batch total for g+r+i

2) Per-candidate scheduled time cost:
      batch_exptime_total + slew_overhead_s

   so if exptime_s = 90, total cost = 90 + 180 = 270 s

3) Observation returned by env:
   (
       gw_scalar_vec,      shape (G_scalar,)
       chirp_mass_bins,    shape (G_bins,)
       events,             shape (K_MAX, N_OBS_MAX, 5)
       cand_mask,          shape (K_MAX,)
       obs_mask,           shape (K_MAX, N_OBS_MAX)
       day                 shape (1,)
   )

4) Event features per row:
   [time_days, mag, mag_sigma, mag_limit, filter_id]

Expected files in each scenario directory
-----------------------------------------
scenario_metadata.csv
true_all.csv
KN_true.csv
observe_all.csv
KN_observe.csv
"""

from __future__ import annotations

import os
import glob
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

from sklearn.neighbors import KernelDensity
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

import astropy.units as u
from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_body
from astroplan import (
    Observer,
    FixedTarget,
    AltitudeConstraint,
    AirmassConstraint,
    MoonSeparationConstraint,
    is_observable,
)


# ============================================================
# Constants
# ============================================================
VALID_CANON_BANDS = ("g", "r", "i")

FILTER_MAP_HIST_TO_CANON = {
    "sdssg": "g",
    "sdssr": "r",
    "sdssi": "i",
    "g": "g",
    "r": "r",
    "i": "i",
}

FILTER_TO_ID = {
    "u": 0,
    "g": 1,
    "r": 2,
    "i": 3,
    "z": 4,
    "y": 5,
}

ACTION_TO_EXPTIME = {
    0: None,
    1: 90,
    2: 120,
    3: 180,
}

ALL_MODEL_FILTERS = tuple(FILTER_TO_ID.keys())


# ============================================================
# General utilities
# ============================================================
def _sigmoid(x: float) -> float:
    return float(1.0 / (1.0 + np.exp(-x)))


def safe_read_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path)


def build_gw_scalar_vector(meta_row: pd.Series) -> np.ndarray:
    vals = [
        float(meta_row.get("chirp_mass", np.nan)),
        float(meta_row.get("HasNS", np.nan)),
        float(meta_row.get("HasRemnant", np.nan)),
        float(meta_row.get("HasMassGap", np.nan)),
        float(meta_row.get("area90", np.nan)),
        float(meta_row.get("distmean", np.nan)),
        float(meta_row.get("diststd", np.nan)),
        float(meta_row.get("ra", np.nan)),
        float(meta_row.get("dec", np.nan)),
    ]
    vals = np.asarray(vals, dtype=np.float32)
    vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
    return vals


def build_chirp_mass_bins(meta_row: pd.Series) -> np.ndarray:
    edges = meta_row.get("chirp_mass_bin_edges", None)

    if isinstance(edges, str):
        try:
            arr = np.fromstring(edges.strip("[]"), sep=",", dtype=np.float32)
            arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            return arr
        except Exception:
            pass

    return np.zeros((0,), dtype=np.float32)


def weighted_linear_fit_result(
    x: np.ndarray,
    y: np.ndarray,
    yerr: np.ndarray,
) -> Optional[Dict[str, float]]:
    """
    Fit y = a + b x with weights 1/yerr^2 and return the best-fit parameters
    plus chi^2 and reduced chi^2.

    chi^2(a,b) = sum_i [ (y_i - (a + b x_i)) / yerr_i ]^2
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    yerr = np.asarray(yerr, float)

    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(yerr) & (yerr > 0)
    x, y, yerr = x[mask], y[mask], yerr[mask]

    if len(x) < 2:
        return None

    w = 1.0 / (yerr ** 2)

    S = np.sum(w)
    Sx = np.sum(w * x)
    Sy = np.sum(w * y)
    Sxx = np.sum(w * x * x)
    Sxy = np.sum(w * x * y)

    denom = S * Sxx - Sx * Sx
    if denom <= 0:
        return None

    a = (Sxx * Sy - Sx * Sxy) / denom
    b = (S * Sxy - Sx * Sy) / denom

    yhat = a + b * x
    chi2 = np.sum(((y - yhat) / yerr) ** 2)
    dof = max(len(x) - 2, 1)
    red_chi2 = chi2 / dof

    return {
        "a": float(a),
        "b": float(b),
        "chi2": float(chi2),
        "reduced_chi2": float(red_chi2),
        "n_points": int(len(x)),
    }


def compute_mag_fit_stats_by_filter(
    df: pd.DataFrame,
    allowed_filters: Tuple[str, ...] = ALL_MODEL_FILTERS,
    min_points_per_filter: int = 2,
) -> Dict[str, Dict[str, float]]:
    """
    For each filter, return:
      - a
      - b
      - chi2
      - reduced_chi2
      - n_points
    """
    out: Dict[str, Dict[str, float]] = {}

    if df is None or df.empty:
        return out

    dff = df.copy()
    dff["filter"] = dff["filter"].astype(str).str.strip().str.lower()
    dff["time"] = pd.to_numeric(dff["time"], errors="coerce")
    dff["mag"] = pd.to_numeric(dff["mag"], errors="coerce")
    dff["magerr"] = pd.to_numeric(dff["magerr"], errors="coerce")

    dff = dff[
        np.isfinite(dff["time"])
        & np.isfinite(dff["mag"])
        & np.isfinite(dff["magerr"])
        & (dff["magerr"] > 0)
    ]
    dff = dff[dff["filter"].isin(allowed_filters)]

    if dff.empty:
        return out

    for band in allowed_filters:
        sub = dff[dff["filter"] == band].sort_values("time")
        if len(sub) < min_points_per_filter:
            continue

        x = sub["time"].to_numpy(float)
        y = sub["mag"].to_numpy(float)
        yerr = sub["magerr"].to_numpy(float)

        fit = weighted_linear_fit_result(x, y, yerr)
        if fit is not None:
            out[str(band)] = fit

    return out


def compute_average_filter_improvement(
    before_map: Dict[str, Dict[str, float]],
    after_map: Dict[str, Dict[str, float]],
    min_filters_required: int = 2,
    use_reduced_chi2: bool = True,
) -> Tuple[float, List[str], Dict[str, float]]:
    """
    Improvement is defined as chi2_before - chi2_after using the minimum chi2
    from the best-fit line in each filter.
    """
    valid_filters = sorted(set(before_map.keys()) & set(after_map.keys()))
    per_filter_improvement: Dict[str, float] = {}

    key = "reduced_chi2" if use_reduced_chi2 else "chi2"

    for f in valid_filters:
        b = float(before_map[f][key])
        a = float(after_map[f][key])
        per_filter_improvement[f] = b - a

    if len(valid_filters) < min_filters_required:
        return 0.0, valid_filters, per_filter_improvement

    avg_improvement = float(np.mean([per_filter_improvement[f] for f in valid_filters]))
    return avg_improvement, valid_filters, per_filter_improvement


# ============================================================
# Truth LC handling
# ============================================================
@dataclass
class TruthLC:
    t_by_band: Dict[str, np.ndarray]
    m_by_band: Dict[str, np.ndarray]

    def mag(self, t: float, band: str) -> float:
        if band not in self.t_by_band:
            return np.nan
        tt = self.t_by_band[band]
        mm = self.m_by_band[band]
        if tt.size == 0:
            return np.nan
        if t <= tt[0]:
            return float(mm[0])
        if t >= tt[-1]:
            return float(mm[-1])
        return float(np.interp(t, tt, mm))


def truth_df_to_truthlc(
    truth_df: pd.DataFrame,
    keep_bands: Tuple[str, ...] = VALID_CANON_BANDS,
) -> TruthLC:
    df = truth_df.copy()
    df["filter"] = df["filter"].astype(str).str.strip().str.lower()
    df["time"] = pd.to_numeric(df["time"], errors="coerce")
    df["mag"] = pd.to_numeric(df["mag"], errors="coerce")
    df = df[np.isfinite(df["time"]) & np.isfinite(df["mag"])]
    df = df[df["filter"].isin(keep_bands)]

    t_by_band = {}
    m_by_band = {}
    for band, gb in df.groupby("filter"):
        tt = gb["time"].to_numpy(float)
        mm = gb["mag"].to_numpy(float)
        order = np.argsort(tt)
        t_by_band[band] = tt[order]
        m_by_band[band] = mm[order]

    return TruthLC(t_by_band=t_by_band, m_by_band=m_by_band)


# ============================================================
# SEDM historical models (conditions + magerr only, no cadence)
# ============================================================
@dataclass
class ConditionModel:
    kdes: Dict[Tuple[int, str], KernelDensity]
    scalers: Dict[Tuple[int, str], StandardScaler]
    skynoise_max: float = 100.0

    def sample(self, exptime_s: int, band: str, rng: np.random.Generator) -> Tuple[float, float]:
        key = (int(exptime_s), str(band))
        if key not in self.kdes:
            raise KeyError(f"No KDE for exptime_s={exptime_s}, band={band}")

        kde = self.kdes[key]
        scaler = self.scalers[key]

        seed = int(rng.integers(0, 2**31 - 1))
        z_scaled = kde.sample(1, random_state=seed).reshape(1, -1)
        z = scaler.inverse_transform(z_scaled).reshape(-1)

        skynoise = float(z[0])
        mlim = float(z[1])

        if not np.isfinite(skynoise) or skynoise <= 0:
            skynoise = 1.0
        skynoise = float(np.clip(skynoise, 1e-6, self.skynoise_max))

        if not np.isfinite(mlim):
            mlim = 20.0

        return skynoise, mlim


def fit_condition_kdes_from_historical(
    hist_csv: str,
    exptimes: Tuple[int, ...] = (90, 120, 180),
    bands: Tuple[str, ...] = VALID_CANON_BANDS,
    skynoise_max: float = 100.0,
    skynoise_jitter_frac: float = 0.02,
    bw_min: float = 0.05,
    bw_max: float = 1.50,
    n_grid: int = 30,
    min_rows: int = 100,
) -> ConditionModel:
    df = pd.read_csv(hist_csv, usecols=["exptime_s", "filter", "skynoise", "limiting_mag"])
    df["exptime_s"] = pd.to_numeric(df["exptime_s"], errors="coerce")
    df["skynoise"] = pd.to_numeric(df["skynoise"], errors="coerce")
    df["limiting_mag"] = pd.to_numeric(df["limiting_mag"], errors="coerce")
    df["filter"] = df["filter"].astype(str).str.strip().str.lower()
    df["filter"] = df["filter"].map(FILTER_MAP_HIST_TO_CANON)

    df = df[np.isfinite(df["exptime_s"]) & np.isfinite(df["skynoise"]) & np.isfinite(df["limiting_mag"])]
    df = df.dropna(subset=["filter"])
    df = df[df["exptime_s"].isin(exptimes)]
    df = df[df["filter"].isin(bands)]
    df = df[(df["skynoise"] > 0) & (df["skynoise"] <= skynoise_max)]

    rng = np.random.default_rng(0)
    kdes = {}
    scalers = {}

    bw_grid = np.logspace(np.log10(bw_min), np.log10(bw_max), n_grid)

    for expt in exptimes:
        for b in bands:
            sub = df[(df["exptime_s"] == expt) & (df["filter"] == b)]
            if len(sub) < min_rows:
                continue

            skynoise = sub["skynoise"].to_numpy(float)
            if skynoise_jitter_frac > 0:
                skynoise = skynoise * (1.0 + rng.normal(0.0, skynoise_jitter_frac, size=skynoise.shape))
            mlim = sub["limiting_mag"].to_numpy(float)

            X = np.column_stack([skynoise, mlim])
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X)

            grid = GridSearchCV(
                KernelDensity(kernel="gaussian"),
                {"bandwidth": bw_grid},
                cv=5,
                n_jobs=-1,
            )
            grid.fit(Xs)

            kdes[(int(expt), str(b))] = grid.best_estimator_
            scalers[(int(expt), str(b))] = scaler

    return ConditionModel(kdes=kdes, scalers=scalers, skynoise_max=skynoise_max)


@dataclass
class MagErrSampler:
    delta_edges: np.ndarray
    bins: Dict[Tuple[int, str], List[np.ndarray]]

    def sample(self, exptime_s: int, band: str, delta: float, rng: np.random.Generator) -> float:
        key = (int(exptime_s), str(band))
        if key not in self.bins:
            return self._fallback(delta, rng)

        edges = self.delta_edges
        j = int(np.clip(np.searchsorted(edges, delta, side="right") - 1, 0, len(edges) - 2))
        arr = self.bins[key][j]

        if arr.size == 0:
            per = self.bins[key]
            for step in range(1, len(per)):
                j1 = max(0, j - step)
                j2 = min(len(per) - 1, j + step)
                if per[j1].size:
                    arr = per[j1]
                    break
                if per[j2].size:
                    arr = per[j2]
                    break

        if arr.size == 0:
            return self._fallback(delta, rng)

        return float(rng.choice(arr))

    @staticmethod
    def _fallback(delta: float, rng: np.random.Generator) -> float:
        base = 0.03 + 0.10 * _sigmoid(delta / 0.5)
        return float(np.clip(rng.normal(base, 0.02), 0.01, 0.3))


def fit_magerr_sampler_from_historical(
    hist_csv: str,
    exptimes: Tuple[int, ...] = (90, 120, 180),
    bands: Tuple[str, ...] = VALID_CANON_BANDS,
    skynoise_max: float = 100.0,
    max_magerr: float = 0.5,
    delta_edges: Optional[np.ndarray] = None,
) -> MagErrSampler:
    if delta_edges is None:
        delta_edges = np.array(
            [-6, -4, -3, -2, -1.5, -1.0, -0.7, -0.5, -0.3, -0.2, -0.1, 0.0, 0.2, 0.5, 1.0],
            dtype=float,
        )

    df = pd.read_csv(hist_csv, usecols=["exptime_s", "filter", "mag", "magerr", "limiting_mag", "skynoise"])
    df["exptime_s"] = pd.to_numeric(df["exptime_s"], errors="coerce")
    df["mag"] = pd.to_numeric(df["mag"], errors="coerce")
    df["magerr"] = pd.to_numeric(df["magerr"], errors="coerce")
    df["limiting_mag"] = pd.to_numeric(df["limiting_mag"], errors="coerce")
    df["skynoise"] = pd.to_numeric(df["skynoise"], errors="coerce")
    df["filter"] = df["filter"].astype(str).str.strip().str.lower()
    df["filter"] = df["filter"].map(FILTER_MAP_HIST_TO_CANON)

    df = df[
        np.isfinite(df["exptime_s"])
        & np.isfinite(df["mag"])
        & np.isfinite(df["magerr"])
        & np.isfinite(df["limiting_mag"])
        & np.isfinite(df["skynoise"])
    ]
    df = df.dropna(subset=["filter"])
    df = df[df["exptime_s"].isin(exptimes)]
    df = df[df["filter"].isin(bands)]
    df = df[(df["skynoise"] > 0) & (df["skynoise"] <= skynoise_max)]
    df = df[(df["magerr"] > 0) & (df["magerr"] <= max_magerr)]

    df["delta"] = df["mag"] - df["limiting_mag"]

    edges = np.asarray(delta_edges, dtype=float)
    nb = len(edges) - 1

    bins = {}
    for expt in exptimes:
        for b in bands:
            sub = df[(df["exptime_s"] == expt) & (df["filter"] == b)]
            d = sub["delta"].to_numpy(float)
            e = sub["magerr"].to_numpy(float)

            per_bin = []
            for i in range(nb):
                m = (d >= edges[i]) & (d < edges[i + 1])
                per_bin.append(e[m])
            bins[(int(expt), str(b))] = per_bin

    return MagErrSampler(delta_edges=edges, bins=bins)


def ensure_condition_fallbacks(
    conditions: ConditionModel,
    exptimes: Tuple[int, ...] = (90, 120, 180),
    bands: Tuple[str, ...] = VALID_CANON_BANDS,
) -> None:
    have = set(conditions.kdes.keys())
    for b in bands:
        avail = sorted([e for (e, bb) in have if bb == b])
        if not avail:
            continue
        for e in exptimes:
            key = (int(e), str(b))
            if key in have:
                continue
            nearest = min(avail, key=lambda x: abs(x - e))
            conditions.kdes[key] = conditions.kdes[(nearest, b)]
            conditions.scalers[key] = conditions.scalers[(nearest, b)]


class SEDMObservationModel:
    """
    Scheduler-only SEDM generator.

    The scheduler decides when the observation happens.
    For each executed request, generate exactly one observation
    in g, r, and i at that scheduled time.

    Historical SEDM is used only for:
    - limiting_mag / skynoise sampling
    - magerr sampling
    NOT for cadence / visit timing.
    """

    def __init__(
        self,
        hist_csv: str,
        exptimes: Tuple[int, ...] = (90, 120, 180),
        bands: Tuple[str, ...] = VALID_CANON_BANDS,
        skynoise_max: float = 100.0,
        max_magerr: float = 0.5,
    ):
        self.conditions = fit_condition_kdes_from_historical(
            hist_csv,
            exptimes=exptimes,
            bands=bands,
            skynoise_max=skynoise_max,
        )
        self.magerr_sampler = fit_magerr_sampler_from_historical(
            hist_csv,
            exptimes=exptimes,
            bands=bands,
            skynoise_max=skynoise_max,
            max_magerr=max_magerr,
        )
        ensure_condition_fallbacks(self.conditions, exptimes=exptimes, bands=bands)
        self.bands = tuple(bands)

    def generate(
        self,
        source_id: str,
        truth_df: pd.DataFrame,
        exptime_s: int,
        t_start: float,
        rng: np.random.Generator,
        max_visits: int = 1,  # kept for API compatibility
    ) -> pd.DataFrame:
        truth_lc = truth_df_to_truthlc(truth_df, keep_bands=self.bands)

        rows = []
        t_obs = float(t_start)

        for band in self.bands:
            m_true = truth_lc.mag(t_obs, band)
            skynoise, mlim = self.conditions.sample(exptime_s, band, rng)

            p_det = 0.0
            if np.isfinite(m_true):
                p_det = _sigmoid((mlim - m_true) / 0.25)

            is_det = bool(rng.random() < p_det)

            mag = np.nan
            magerr = np.nan

            if is_det and np.isfinite(m_true):
                delta = float(m_true - mlim)
                magerr = self.magerr_sampler.sample(exptime_s, band, delta, rng)
                mag = float(rng.normal(m_true, magerr))

            rows.append(
                {
                    "source_id": str(source_id),
                    "time": t_obs,
                    "filter": str(band),
                    "mag": mag,
                    "magerr": magerr,
                    "limiting_mag": float(mlim),
                }
            )

        return pd.DataFrame(rows)


# ============================================================
# Scheduler data classes
# ============================================================
@dataclass
class FollowupRequest:
    request_id: str
    candidate_id: str
    slot_idx: int
    exptime_s: int
    request_day: int
    priority: float
    ra: float
    dec: float
    status: str = "PENDING"
    scheduled_time: Optional[float] = None
    executed_day: Optional[int] = None
    reason: str = ""


@dataclass
class CandidateState:
    source_id: str
    slot_idx: int
    cand_type: str
    ra: float
    dec: float
    z: float
    is_kn: bool = False
    observed_rows: List[dict] = field(default_factory=list)

    mag_fit_before: Dict[str, float] = field(default_factory=dict)
    mag_fit_after: Dict[str, float] = field(default_factory=dict)

    def add_rows(self, rows: List[dict]) -> None:
        self.observed_rows.extend(rows)

    def observed_df(self) -> pd.DataFrame:
        if not self.observed_rows:
            return pd.DataFrame(columns=["time", "filter", "mag", "magerr", "limmag", "origin"])
        return pd.DataFrame(self.observed_rows).sort_values(["time", "filter"]).reset_index(drop=True)


# ============================================================
# Scheduler
# ============================================================
class SimplifiedSEDMScheduler:
    def __init__(
        self,
        location: Optional[EarthLocation] = None,
        slew_overhead_s: float = 180.0,
        focus_time_s: float = 300.0,
        standard_time_s: float = 300.0,
        nightly_budget_hours: float = 5.0,
        altitude_min_deg: float = 15.0,
        airmass_range: Tuple[float, float] = (1.0, 2.8),
    ):
        if location is None:
            location = EarthLocation(
                lat=33.3563 * u.deg,
                lon=-116.8648 * u.deg,
                height=1706 * u.m,
            )

        self.location = location
        self.observer = Observer(location=self.location, name="Palomar", timezone="UTC")

        self.slew_overhead_s = float(slew_overhead_s)
        self.focus_time_s = float(focus_time_s)
        self.standard_time_s = float(standard_time_s)
        self.nightly_budget_s = float(nightly_budget_hours) * 3600.0

        self.altitude_min_deg = float(altitude_min_deg)
        self.airmass_range = airmass_range

    def _night_bounds(self, base_date: Time) -> Tuple[Time, Time]:
        start_time = self.observer.twilight_evening_nautical(base_date, which="next")
        end_time = self.observer.twilight_morning_astronomical(start_time, which="next")
        return start_time, end_time

    def build_target_table(self, requests: List[FollowupRequest], current_time: Time) -> pd.DataFrame:
        rows = []
        for req in requests:
            if req.status != "PENDING":
                continue

            coord = SkyCoord(ra=req.ra * u.deg, dec=req.dec * u.deg)
            fixed_object = FixedTarget(coord=coord, name=req.candidate_id)

            obs_total = req.exptime_s + self.slew_overhead_s

            rows.append(
                {
                    "request_id": req.request_id,
                    "candidate_id": req.candidate_id,
                    "slot_idx": req.slot_idx,
                    "priority": float(req.priority),
                    "objname": req.candidate_id,
                    "ra": float(req.ra),
                    "dec": float(req.dec),
                    "fixed_object": fixed_object,
                    "SkyCoords": coord,
                    "obs_seq": {
                        "ifu_exptime": 0,
                        "rc": True,
                        "rc_obs_dict": {"obs_order": "gri", "obs_exptime": req.exptime_s},
                        "total": float(obs_total),
                    },
                    "maxairmass": self.airmass_range[1],
                    "typedesig": "f",
                }
            )

        if len(rows) == 0:
            return pd.DataFrame()

        targets = pd.DataFrame(rows)
        targets = self.update_targets_coords(targets, current_time)
        return targets

    def update_targets_coords(self, target_list: pd.DataFrame, obsdatetime: Time) -> pd.DataFrame:
        if len(target_list) == 0:
            return target_list

        out = target_list.copy()

        start_alt = []
        end_alt = []
        start_airmass = []
        end_airmass = []
        set_time = []

        for row in out.itertuples():
            coord = row.SkyCoords

            start_altaz = coord.transform_to(AltAz(obstime=obsdatetime, location=self.location))
            finish = obsdatetime + TimeDelta(row.obs_seq["total"], format="sec")
            end_altaz = coord.transform_to(AltAz(obstime=finish, location=self.location))

            try:
                target_set = self.observer.target_set_time(
                    obsdatetime,
                    row.fixed_object,
                    which="next",
                    horizon=0 * u.deg,
                )
                target_set_val = float(target_set.jd)
            except Exception:
                fallback_set = obsdatetime + TimeDelta(10 * 3600, format="sec")
                target_set_val = float(fallback_set.jd)

            start_alt.append(float(start_altaz.alt.deg))
            end_alt.append(float(end_altaz.alt.deg))

            sa = float(start_altaz.secz) if np.isfinite(start_altaz.secz) else np.nan
            ea = float(end_altaz.secz) if np.isfinite(end_altaz.secz) else np.nan
            start_airmass.append(sa)
            end_airmass.append(ea)
            set_time.append(target_set_val)

        out["start_alt"] = start_alt
        out["end_alt"] = end_alt
        out["start_airmass"] = start_airmass
        out["end_airmass"] = end_airmass
        out["set_time"] = set_time

        return out

    def get_next_observable_target(
        self,
        target_list: pd.DataFrame,
        obsdatetime: Time,
        remaining_budget_s: float,
        do_airmass: bool = True,
        do_moon_sep: bool = True,
        sort_columns: Tuple[str, ...] = ("priority", "set_time"),
        sort_order: Tuple[bool, ...] = (False, False),
        update_coords: bool = True,
    ) -> Tuple[Optional[str], Optional[dict]]:
        if len(target_list) == 0:
            return None, None

        if update_coords:
            target_list = self.update_targets_coords(target_list, obsdatetime)

        target_list = target_list.sort_values(list(sort_columns), ascending=list(sort_order))

        for row in target_list.itertuples():
            total_time_s = float(row.obs_seq["total"])
            if total_time_s > remaining_budget_s:
                continue

            start = obsdatetime
            finish = start + TimeDelta(total_time_s, format="sec")

            constraints = [AltitudeConstraint(min=self.altitude_min_deg * u.deg)]

            if do_airmass:
                constraints.append(AirmassConstraint(min=self.airmass_range[0], max=row.maxairmass))

            moon = get_body("moon", start, location=self.location)
            moon_dist = moon.separation(row.SkyCoords).deg

            if do_moon_sep:
                moon_illum = self.observer.moon_illumination(start) * 100.0
                min_moon_sep = 5.0 + max(0.0, moon_illum - 75.0)
                constraints.append(MoonSeparationConstraint(min=min_moon_sep * u.deg))

            observable = is_observable(
                constraints,
                self.observer,
                row.fixed_object,
                times=[start, finish],
                time_grid_resolution=0.1 * u.hour,
            )

            if not observable:
                continue

            s_air = float(row.start_airmass) if np.isfinite(row.start_airmass) else 99.0
            e_air = float(row.end_airmass) if np.isfinite(row.end_airmass) else 99.0

            if s_air > 3.5 or e_air > 3.5:
                continue

            chosen = {
                "request_id": row.request_id,
                "candidate_id": row.candidate_id,
                "slot_idx": row.slot_idx,
                "start_time": start,
                "finish_time": finish,
                "obs_seq": row.obs_seq,
                "moon_dist": float(moon_dist),
                "start_airmass": s_air,
                "end_airmass": e_air,
                "start_alt": float(row.start_alt),
                "end_alt": float(row.end_alt),
                "duration_s": total_time_s,
            }
            return row.request_id, chosen

        return None, None

    def simulate_night(
        self,
        requests: List[FollowupRequest],
        night_date: Time,
        do_focus: bool = True,
        do_standard: bool = True,
        sort_columns: Tuple[str, ...] = ("priority", "start_alt"),
        sort_order: Tuple[bool, ...] = (False, False),
    ) -> List[dict]:
        start_time, twilight_end = self._night_bounds(night_date)

        budget_end = start_time + TimeDelta(self.nightly_budget_s, format="sec")
        end_time = min(twilight_end, budget_end)

        targets = self.build_target_table(requests, start_time)
        if len(targets) == 0:
            return []

        current_time = start_time
        executed = []

        if do_focus:
            current_time += TimeDelta(self.focus_time_s, format="sec")
            do_focus = False

        if do_standard:
            current_time += TimeDelta(self.standard_time_s, format="sec")
            do_standard = False

        while current_time < end_time and len(targets) > 0:
            remaining_budget_s = (end_time - current_time).sec
            if remaining_budget_s <= 0:
                break

            targets = self.update_targets_coords(targets, current_time)
            targets = targets.sort_values(list(sort_columns), ascending=list(sort_order))

            req_id, chosen = self.get_next_observable_target(
                targets,
                obsdatetime=current_time,
                remaining_budget_s=remaining_budget_s,
                do_airmass=True,
                do_moon_sep=True,
                sort_columns=("priority", "set_time"),
                sort_order=(False, False),
                update_coords=False,
            )

            if req_id is None:
                current_time += TimeDelta(300, format="sec")
                continue

            executed.append(chosen)
            targets = targets[targets.request_id != req_id]
            current_time += TimeDelta(chosen["duration_s"], format="sec")

        return executed


# ============================================================
# Main environment
# ============================================================
class KilonovaSEDMEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        scenario_dirs: List[str],
        hist_sedm_csv: str,
        k_max: int,
        n_obs_max: int,
        n_days: int = 7,
        initial_days_revealed: float = 2.0,
        expire_after_days: int = 7,
        non_execution_penalty: float = 0.0,
        episode_start_time: str = "2026-01-01 00:00:00",
        min_filters_for_reward: int = 1,
        reward_filters: Tuple[str, ...] = ALL_MODEL_FILTERS,
        nightly_budget_hours: float = 5.0,
        slew_overhead_s: float = 180.0,
        seed: Optional[int] = None,
    ):
        super().__init__()

        if len(scenario_dirs) == 0:
            raise ValueError("scenario_dirs is empty")

        self.scenario_dirs = scenario_dirs
        self.hist_sedm_csv = hist_sedm_csv
        self.K_MAX = int(k_max)
        self.N_OBS_MAX = int(n_obs_max)
        self.N_DAYS = int(n_days)
        self.initial_days_revealed = float(initial_days_revealed)
        self.expire_after_days = int(expire_after_days)
        self.non_execution_penalty = float(non_execution_penalty)
        self.min_filters_for_reward = int(min_filters_for_reward)
        self.reward_filters = tuple(reward_filters)

        self.nightly_budget_hours = float(nightly_budget_hours)
        self.slew_overhead_s = float(slew_overhead_s)

        self.rng = np.random.default_rng(seed)
        self.ref_time = Time(episode_start_time, scale="utc")

        self.sedm_model = SEDMObservationModel(hist_csv=self.hist_sedm_csv)
        self.scheduler = SimplifiedSEDMScheduler(
            slew_overhead_s=self.slew_overhead_s,
            nightly_budget_hours=self.nightly_budget_hours,
        )

        self.G_scalar, self.G_bins = self._infer_g_dims()

        self.observation_space = spaces.Tuple((
            spaces.Box(-np.inf, np.inf, shape=(self.G_scalar,), dtype=np.float32),
            spaces.Box(-np.inf, np.inf, shape=(self.G_bins,), dtype=np.float32),
            spaces.Box(-np.inf, np.inf, shape=(self.K_MAX, self.N_OBS_MAX, 5), dtype=np.float32),
            spaces.MultiBinary(self.K_MAX),
            spaces.MultiBinary((self.K_MAX, self.N_OBS_MAX)),
            spaces.Box(0.0, float(self.N_DAYS), shape=(1,), dtype=np.float32),
        ))

        self.action_space = spaces.MultiDiscrete([4] * self.K_MAX)

        self.current_day = 0
        self.scenario_dir: Optional[str] = None

        self.gw_scalar_vec: Optional[np.ndarray] = None
        self.chirp_mass_bins: Optional[np.ndarray] = None

        self.candidates: Dict[str, CandidateState] = {}
        self.slot_to_source: Dict[int, str] = {}
        self.source_to_slot: Dict[str, int] = {}

        self.truth_map: Dict[str, pd.DataFrame] = {}
        self.rubin_obs_map: Dict[str, pd.DataFrame] = {}

        self.pending_requests: List[FollowupRequest] = []
        self.requested_last_step: List[FollowupRequest] = []
        self.executed_last_step: List[FollowupRequest] = []

        self._obs_tensor: Optional[np.ndarray] = None
        self._obs_mask: Optional[np.ndarray] = None
        self._cand_mask: Optional[np.ndarray] = None

        self.last_rubin_reveal_time: float = 0.0
        self._request_counter: int = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        self.current_day = 0
        self.pending_requests = []
        self.requested_last_step = []
        self.executed_last_step = []
        self._request_counter = 0

        self._load_random_scenario()
        self._initialize_buffers()

        self.last_rubin_reveal_time = float(self.initial_days_revealed)
        self._reveal_rubin_data_until(self.last_rubin_reveal_time)

        obs = self._build_observation()
        info = self._build_info(initial=True)
        return obs, info

    def step(self, action):
        action = np.asarray(action, dtype=int)
        if action.shape != (self.K_MAX,):
            raise ValueError(f"Expected action shape {(self.K_MAX,)}, got {action.shape}")

        before_fit_maps = self._compute_followup_filter_fit_maps(action)
        self.requested_last_step = self._enqueue_requests_from_action(action)

        night_date = self.ref_time + TimeDelta(self.current_day * 86400, format="sec")
        self.executed_last_step = self._run_scheduler_one_night(night_date)
        self._materialize_executed_requests(self.executed_last_step)

        # Reward is based on the effect of the executed follow-up itself.
        after_fit_maps = self._compute_followup_filter_fit_maps_from_requests(self.requested_last_step)

        reward, reward_breakdown = self._compute_reward(
            requested=self.requested_last_step,
            executed=self.executed_last_step,
            before_fit_maps=before_fit_maps,
            after_fit_maps=after_fit_maps,
        )

        self.current_day += 1

        new_t_hi = self.initial_days_revealed + self.current_day
        self._reveal_rubin_data_between(self.last_rubin_reveal_time, new_t_hi)
        self.last_rubin_reveal_time = new_t_hi

        obs = self._build_observation()
        terminated = self._termination_condition()
        truncated = bool(self.current_day >= self.N_DAYS and not terminated)

        info = self._build_info(
            initial=False,
            reward_breakdown=reward_breakdown,
            requested=self.requested_last_step,
            executed=self.executed_last_step,
        )
        return obs, reward, terminated, truncated, info


    def render(self):
        print(f"Day: {self.current_day}")
        print(f"Scenario: {self.scenario_dir}")
        print(f"Candidates: {len(self.candidates)}")
        print(f"Pending requests: {sum(r.status == 'PENDING' for r in self.pending_requests)}")

    # --------------------------------------------------------
    # Scenario handling
    # --------------------------------------------------------
    def _infer_g_dims(self) -> Tuple[int, int]:
        meta_path = os.path.join(self.scenario_dirs[0], "scenario_metadata.csv")
        meta = safe_read_csv(meta_path)
        row = meta.iloc[0]

        g_scalar = len(build_gw_scalar_vector(row))
        g_bins = len(build_chirp_mass_bins(row))
        return g_scalar, g_bins

    def _load_random_scenario(self) -> None:
        self.scenario_dir = str(self.rng.choice(self.scenario_dirs))
        self._load_scenario(self.scenario_dir)

    def _load_scenario(self, scenario_dir: str) -> None:
        meta = safe_read_csv(os.path.join(scenario_dir, "scenario_metadata.csv"))
        true_all = safe_read_csv(os.path.join(scenario_dir, "true_all.csv"))
        obs_all = safe_read_csv(os.path.join(scenario_dir, "observe_all.csv"))

        row = meta.iloc[0]
        self.gw_scalar_vec = build_gw_scalar_vector(row).astype(np.float32)
        self.chirp_mass_bins = build_chirp_mass_bins(row).astype(np.float32)

        for df in [true_all, obs_all]:
            df["source_id"] = df["source_id"].astype(str)
            df["filter"] = df["filter"].astype(str).str.strip().str.lower()
            df["time"] = pd.to_numeric(df["time"], errors="coerce")
            df["ra"] = pd.to_numeric(df["ra"], errors="coerce")
            df["dec"] = pd.to_numeric(df["dec"], errors="coerce")
            df["z"] = pd.to_numeric(df["z"], errors="coerce")
            if "mag" in df.columns:
                df["mag"] = pd.to_numeric(df["mag"], errors="coerce")
            if "magerr" in df.columns:
                df["magerr"] = pd.to_numeric(df["magerr"], errors="coerce")
            if "limmag" in df.columns:
                df["limmag"] = pd.to_numeric(df["limmag"], errors="coerce")

        source_ids = sorted(true_all["source_id"].unique().tolist())
        if len(source_ids) > self.K_MAX:
            raise ValueError(f"Scenario has {len(source_ids)} candidates but K_MAX={self.K_MAX}")

        self.candidates = {}
        self.slot_to_source = {}
        self.source_to_slot = {}

        for slot_idx, sid in enumerate(source_ids):
            rows = true_all[true_all["source_id"] == sid]
            first = rows.iloc[0]
            cand_type = str(first.get("type", "unknown"))
            is_kn = cand_type.upper() == "KN"

            cand = CandidateState(
                source_id=sid,
                slot_idx=slot_idx,
                cand_type=cand_type,
                ra=float(first["ra"]),
                dec=float(first["dec"]),
                z=float(first["z"]),
                is_kn=is_kn,
            )
            self.candidates[sid] = cand
            self.slot_to_source[slot_idx] = sid
            self.source_to_slot[sid] = slot_idx

        self.truth_map = {
            sid: g.copy().sort_values(["time", "filter"]).reset_index(drop=True)
            for sid, g in true_all.groupby("source_id")
        }
        self.rubin_obs_map = {
            sid: g.copy().sort_values(["time", "filter"]).reset_index(drop=True)
            for sid, g in obs_all.groupby("source_id")
        }

    # --------------------------------------------------------
    # Observation tensor handling
    # --------------------------------------------------------
    def _initialize_buffers(self) -> None:
        self._obs_tensor = np.full((self.K_MAX, self.N_OBS_MAX, 5), np.nan, dtype=np.float32)
        self._obs_mask = np.zeros((self.K_MAX, self.N_OBS_MAX), dtype=np.int8)
        self._cand_mask = np.zeros((self.K_MAX,), dtype=np.int8)

        for slot_idx in self.slot_to_source:
            self._cand_mask[slot_idx] = 1

        for cand in self.candidates.values():
            cand.observed_rows = []
            cand.mag_fit_before = {}
            cand.mag_fit_after = {}

    def _build_observation(self):
        return (
            self.gw_scalar_vec.astype(np.float32),
            self.chirp_mass_bins.astype(np.float32),
            self._obs_tensor.astype(np.float32),
            self._cand_mask.astype(np.int8),
            self._obs_mask.astype(np.int8),
            np.asarray([self.current_day], dtype=np.float32),
        )

    def _append_observation_row(self, slot_idx: int, row: dict) -> None:
        free = np.where(self._obs_mask[slot_idx] == 0)[0]
        if len(free) == 0:
            self._obs_tensor[slot_idx, :-1] = self._obs_tensor[slot_idx, 1:]
            self._obs_mask[slot_idx, :-1] = self._obs_mask[slot_idx, 1:]
            j = self.N_OBS_MAX - 1
        else:
            j = int(free[0])

        filt = str(row["filter"]).lower()
        if filt not in FILTER_TO_ID:
            return

        arr = np.array([
            float(row.get("time", np.nan)),
            float(row.get("mag", np.nan)),
            float(row.get("magerr", np.nan)),
            float(row.get("limmag", np.nan)),
            float(FILTER_TO_ID[filt]),
        ], dtype=np.float32)

        self._obs_tensor[slot_idx, j] = arr
        self._obs_mask[slot_idx, j] = 1

    # --------------------------------------------------------
    # Rubin reveal
    # --------------------------------------------------------
    def _reveal_rubin_data_until(self, t_max: float) -> None:
        for sid, df in self.rubin_obs_map.items():
            rows = df[df["time"] <= t_max].to_dict("records")
            self._insert_new_rows(sid, rows, origin="Rubin")

    def _reveal_rubin_data_between(self, t_lo: float, t_hi: float) -> None:
        for sid, df in self.rubin_obs_map.items():
            rows = df[(df["time"] > t_lo) & (df["time"] <= t_hi)].to_dict("records")
            self._insert_new_rows(sid, rows, origin="Rubin")

    def _insert_new_rows(self, source_id: str, rows: List[dict], origin: str) -> None:
        if source_id not in self.candidates or len(rows) == 0:
            return

        cand = self.candidates[source_id]
        slot_idx = cand.slot_idx

        formatted = []
        for r in rows:
            rr = {
                "time": float(r.get("time", np.nan)),
                "filter": str(r.get("filter", "")).lower(),
                "mag": float(r.get("mag", np.nan)),
                "magerr": float(r.get("magerr", np.nan)) if "magerr" in r else np.nan,
                "limmag": float(r.get("limmag", np.nan)) if "limmag" in r else np.nan,
                "origin": origin,
            }
            formatted.append(rr)
            self._append_observation_row(slot_idx, rr)

        cand.add_rows(formatted)

    # --------------------------------------------------------
    # Requests and scheduling
    # --------------------------------------------------------
    def _candidate_has_pending_request(self, sid: str) -> bool:
        return any(r.candidate_id == sid and r.status == "PENDING" for r in self.pending_requests)

    def _compute_followup_filter_fit_maps(self, action: np.ndarray) -> Dict[str, Dict[str, Dict[str, float]]]:
        out: Dict[str, Dict[str, Dict[str, float]]] = {}
        for slot_idx, a in enumerate(action):
            sid = self.slot_to_source.get(slot_idx)
            if sid is None or int(a) == 0:
                continue
            out[sid] = compute_mag_fit_stats_by_filter(
                self.candidates[sid].observed_df(),
                allowed_filters=self.reward_filters,
                min_points_per_filter=2,
            )
        return out

    def _compute_followup_filter_fit_maps_from_requests(
        self,
        requests: List[FollowupRequest],
    ) -> Dict[str, Dict[str, Dict[str, float]]]:
        out: Dict[str, Dict[str, Dict[str, float]]] = {}
        seen = set()
        for req in requests:
            sid = req.candidate_id
            if sid in seen:
                continue
            seen.add(sid)
            out[sid] = compute_mag_fit_stats_by_filter(
                self.candidates[sid].observed_df(),
                allowed_filters=self.reward_filters,
                min_points_per_filter=2,
            )
        return out


    def _enqueue_requests_from_action(self, action: np.ndarray) -> List[FollowupRequest]:
        requests = []
        for slot_idx, a in enumerate(action):
            a = int(a)
            sid = self.slot_to_source.get(slot_idx)
            if sid is None or a == 0:
                continue

            if self._candidate_has_pending_request(sid):
                continue

            exptime = ACTION_TO_EXPTIME[a]
            cand = self.candidates[sid]

            current_fit_map = compute_mag_fit_stats_by_filter(
                cand.observed_df(),
                allowed_filters=self.reward_filters,
                min_points_per_filter=2,
            )

            if len(current_fit_map) > 0:
                mean_chi2 = float(np.mean([v["reduced_chi2"] for v in current_fit_map.values()]))
            else:
                mean_chi2 = 10.0

            priority = 1.0 / (1.0 + mean_chi2)


            req = FollowupRequest(
                request_id=f"req_{self.current_day}_{self._request_counter}_{sid}",
                candidate_id=sid,
                slot_idx=slot_idx,
                exptime_s=int(exptime),
                request_day=self.current_day,
                priority=float(priority),
                ra=cand.ra,
                dec=cand.dec,
            )
            self._request_counter += 1

            self.pending_requests.append(req)
            requests.append(req)

        return requests

    def _run_scheduler_one_night(self, night_date: Time) -> List[FollowupRequest]:
        for req in self.pending_requests:
            if req.status == "PENDING" and (self.current_day - req.request_day) > self.expire_after_days:
                req.status = "EXPIRED"
                req.reason = "expired"

        pending = [r for r in self.pending_requests if r.status == "PENDING"]
        if len(pending) == 0:
            return []

        executed_sched = self.scheduler.simulate_night(
            requests=pending,
            night_date=night_date,
            do_focus=True,
            do_standard=True,
            sort_columns=("priority", "start_alt"),
            sort_order=(False, False),
        )

        executed_by_request_id = {e["request_id"]: e for e in executed_sched}
        executed_requests = []

        for req in pending:
            if req.request_id in executed_by_request_id:
                entry = executed_by_request_id[req.request_id]
                abs_start = entry["start_time"]
                rel_days = (abs_start - self.ref_time).sec / 86400.0

                req.status = "EXECUTED"
                req.executed_day = self.current_day
                req.scheduled_time = float(rel_days)
                req.reason = "scheduled"
                executed_requests.append(req)

        return executed_requests

    # --------------------------------------------------------
    # SEDM materialization
    # --------------------------------------------------------
    def _materialize_executed_requests(self, executed_requests: List[FollowupRequest]) -> None:
        for req in executed_requests:
            sid = req.candidate_id
            truth_df = self.truth_map[sid]

            sedm_df = self.sedm_model.generate(
                source_id=sid,
                truth_df=truth_df,
                exptime_s=req.exptime_s,
                t_start=float(req.scheduled_time),
                rng=self.rng,
                max_visits=1,
            )

            rows = []
            for _, r in sedm_df.iterrows():
                rows.append({
                    "time": float(r["time"]),
                    "filter": str(r["filter"]).lower(),
                    "mag": float(r["mag"]) if np.isfinite(r["mag"]) else np.nan,
                    "magerr": float(r["magerr"]) if np.isfinite(r["magerr"]) else np.nan,
                    "limmag": float(r["limiting_mag"]) if np.isfinite(r["limiting_mag"]) else np.nan,
                    "origin": "SEDM",
                })

            self._insert_new_rows(sid, rows, origin="SEDM")

    # --------------------------------------------------------
    # Reward
    # --------------------------------------------------------
    def _compute_reward(
        self,
        requested: List[FollowupRequest],
        executed: List[FollowupRequest],
        before_fit_maps: Dict[str, Dict[str, Dict[str, float]]],
        after_fit_maps: Dict[str, Dict[str, Dict[str, float]]],
    ) -> Tuple[float, Dict[str, Any]]:
        reward = 0.0
        executed_ids = set(r.candidate_id for r in executed)
        per_candidate = {}
        seen = set()

        for req in requested:
            sid = req.candidate_id
            if sid in seen:
                continue
            seen.add(sid)

            before_map = before_fit_maps.get(sid, {})
            after_map = after_fit_maps.get(sid, {})

            avg_improvement, valid_filters, per_filter_improvement = compute_average_filter_improvement(
                before_map=before_map,
                after_map=after_map,
                min_filters_required=self.min_filters_for_reward,
                use_reduced_chi2=True,
            )

            contrib = float(avg_improvement)
            if sid not in executed_ids:
                contrib -= self.non_execution_penalty

            reward += contrib
            per_candidate[sid] = {
                "fit_before": before_map,
                "fit_after": after_map,
                "valid_filters_used": valid_filters,
                "n_valid_filters": len(valid_filters),
                "avg_improvement": avg_improvement,
                "per_filter_improvement": per_filter_improvement,
                "executed": sid in executed_ids,
                "reward_contribution": contrib,
            }

            self.candidates[sid].mag_fit_before = {
                f: {"reduced_chi2": before_map[f]["reduced_chi2"], "a": before_map[f]["a"], "b": before_map[f]["b"]}
                for f in before_map
            }
            self.candidates[sid].mag_fit_after = {
                f: {"reduced_chi2": after_map[f]["reduced_chi2"], "a": after_map[f]["a"], "b": after_map[f]["b"]}
                for f in after_map
            }

        return float(reward), {
            "requested_ids": [r.candidate_id for r in requested],
            "executed_ids": list(executed_ids),
            "per_candidate": per_candidate,
        }


    # --------------------------------------------------------
    # Termination
    # --------------------------------------------------------
    def _termination_condition(self) -> bool:
        scores = []
        for sid, cand in self.candidates.items():
            fit_map = compute_mag_fit_stats_by_filter(
                cand.observed_df(),
                allowed_filters=self.reward_filters,
                min_points_per_filter=2,
            )
            if len(fit_map) >= self.min_filters_for_reward:
                mean_chi2 = float(np.mean([v["reduced_chi2"] for v in fit_map.values()]))
                scores.append((sid, mean_chi2, cand.is_kn))

        if len(scores) == 0:
            return False

        scores = sorted(scores, key=lambda x: x[1])
        best_sid, best_mean_chi2, best_is_kn = scores[0]

        if not best_is_kn:
            return False

        if len(scores) == 1:
            return True

        second_mean_chi2 = scores[1][1]
        return (second_mean_chi2 - best_mean_chi2) > 0.5


    # --------------------------------------------------------
    # Info
    # --------------------------------------------------------
    def _build_info(
        self,
        initial: bool = False,
        reward_breakdown: Optional[dict] = None,
        requested: Optional[List[FollowupRequest]] = None,
        executed: Optional[List[FollowupRequest]] = None,
    ) -> Dict[str, Any]:

        info = {
            "current_day": self.current_day,
            "n_candidates": len(self.candidates),
            "n_pending_requests": sum(r.status == "PENDING" for r in self.pending_requests),
        }

        if initial:
            return info

        requested = requested or []
        executed = executed or []

        info["requested"] = [
            {
                "candidate_id": r.candidate_id,
                "exptime_s": r.exptime_s,
                "status": r.status,
            }
            for r in requested
        ]

        executed_summary = []
        for r in executed:
            sid = r.candidate_id
            cand_df = self.candidates[sid].observed_df()

            sedm_df = cand_df[cand_df["origin"] == "SEDM"].copy()
            sedm_df["mag"] = pd.to_numeric(sedm_df["mag"], errors="coerce")

            det_df = sedm_df[np.isfinite(sedm_df["mag"])].copy()

            executed_summary.append(
                {
                    "candidate_id": sid,
                    "exptime_s": r.exptime_s,
                    "scheduled_time": r.scheduled_time,
                    "n_sedm_points": int(len(sedm_df)),
                    "n_sedm_detections": int(len(det_df)),
                    "detected_filters": sorted(det_df["filter"].astype(str).unique().tolist()),
                    "has_sedm_detection": bool(len(det_df) > 0),
                }
            )

        info["executed"] = executed_summary
        info["reward"] = reward_breakdown or {}

        return info



# ============================================================
# Example usage
# ============================================================
if __name__ == "__main__":
    SCENARIO_ROOT = "/projects/bdws/npletskova1/git/Pythia/pythia/train_data_gen/BNS_O5a_combined"
    HIST_SEDM_CSV = "/projects/bdws/npletskova1/git/Pythia/pythia/train_data_gen/ZTF/all_sedm_phot_20260220_with_skynoise.csv"

    scenario_dirs = sorted(glob.glob(os.path.join(SCENARIO_ROOT, "scenario_*")))
    print(f"Found {len(scenario_dirs)} scenarios")

    env = KilonovaSEDMEnv(
        scenario_dirs=scenario_dirs,
        hist_sedm_csv=HIST_SEDM_CSV,
        k_max=8498,
        n_obs_max=128,
        n_days=7,
        initial_days_revealed=2.0,
        expire_after_days=6,
        non_execution_penalty=0,
        episode_start_time="2026-01-01 00:00:00",
        min_filters_for_reward=1,
        reward_filters=("g", "r", "i", "u", "z", "y"),
        nightly_budget_hours=5.0,
        slew_overhead_s=180.0,
        seed=42,
    )

    obs, info = env.reset()
    print("Reset info:", info)

    for day in range(env.N_DAYS):
        action = np.zeros(env.K_MAX, dtype=int)

        n = info["n_candidates"]

        if n > 0:
            action[np.random.randint(0, n)] = 3
        if n > 1:
            action[np.random.randint(0, n)] = 2

        obs, reward, terminated, truncated, info = env.step(action)

        print(f"\nDay {day}")
        print("Reward:", reward)
        print("Current day:", info["current_day"])
        print("Requested:", info["requested"])
        print("Executed:", info["executed"])

        if len(info["executed"]) > 0:
            print("Executed candidate details:")
            for ex in info["executed"]:
                print(
                    f"  {ex['candidate_id']}: "
                    f"exptime={ex['exptime_s']} s, "
                    f"scheduled_time={ex['scheduled_time']:.4f}, "
                    f"n_sedm_points={ex['n_sedm_points']}, "
                    f"n_sedm_detections={ex['n_sedm_detections']}, "
                    f"detected_filters={ex['detected_filters']}, "
                    f"has_sedm_detection={ex['has_sedm_detection']}"
                )

        if terminated or truncated:
            print("Episode ended.")
            break

Found 161 scenarios
Reset info: {'current_day': 0, 'n_candidates': 7335, 'n_pending_requests': 0}


 (5167887.79509686, 1356053.320089  , 3474835.47565296)] m, obsgeovel=[(-88.96776767, 378.67007376, 0.213723  ),
 (-98.8766404 , 376.2043155 , 0.23893853)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5136992.73341934, 1468581.16370829, 3474910.28299867)] m, obsgeovel=[( -98.8766404 , 376.2043155 , 0.23893853),
 (-107.08227392, 373.95140488, 0.25982647)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5103643.57053592, 1580406.26264557, 3474991.33846544)] m, obsgeovel=[(-107.08227392, 373.95140488, 0.25982647),
 (-115.23666264, 371.51953778, 0.28059009)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5067856.26587262, 1691475.10239409, 3475078.60327337)] m, obsgeovel=[(-115.23666264, 371.51953778, 0.28059009),
 (-123.33590423, 368.90987801, 0.30121948)] m / s)>. Angular separation can depen


Day 0
Reward: 0.0
Current day: 1
Requested: [{'candidate_id': 'IIn_1790', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'SLSN-I_4402', 'exptime_s': 180, 'status': 'EXECUTED'}]
Executed: [{'candidate_id': 'SLSN-I_4402', 'exptime_s': 180, 'scheduled_time': 0.08243704349216491, 'n_sedm_points': 3, 'n_sedm_detections': 0, 'detected_filters': [], 'has_sedm_detection': False}]
Executed candidate details:
  SLSN-I_4402: exptime=180 s, scheduled_time=0.0824, n_sedm_points=3, n_sedm_detections=0, detected_filters=[], has_sedm_detection=False


 (5139400.5500722 , 1460154.21010579, 3474901.16710098)] m, obsgeovel=[( -96.61084589, 376.79254582, 0.23328287),
 (-106.46780284, 374.12681372, 0.25837924)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5145739.48925364, 1437692.69816752, 3474885.79191652)] m, obsgeovel=[( -96.61084589, 376.79254582, 0.23328287),
 (-104.82988883, 374.58905795, 0.2542084 )] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5106235.13902466, 1572033.94578179, 3474981.78855535)] m, obsgeovel=[(-104.82988883, 374.58905795, 0.2542084 ),
 (-114.62617557, 371.708346  , 0.27915764)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5113063.89645104, 1549716.47058921, 3474965.16588084)] m, obsgeovel=[(-104.82988883, 374.58905795, 0.2542084 ),
 (-112.99876485, 372.20630845, 0.27501232)] m / s)>. Angular separation can dep


Day 1
Reward: 0.0
Current day: 2
Requested: [{'candidate_id': 'II_1229', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'Ic-BL_3705', 'exptime_s': 180, 'status': 'PENDING'}]
Executed: []


...


Day 4
Reward: 0.0
Current day: 5
Requested: [{'candidate_id': 'SLSN-I_4428', 'exptime_s': 180, 'status': 'PENDING'}, {'candidate_id': 'SLSN-I_5058', 'exptime_s': 120, 'status': 'PENDING'}]
Executed: []


 (5003436.02839986, 1873205.52761342, 3475220.19640779)] m, obsgeovel=[(-126.98058862, 367.67126701, 0.31060611),
 (-136.58766133, 364.21151191, 0.33511787)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5011583.45563942, 1851334.97773876, 3475200.21291638)] m, obsgeovel=[(-126.98058862, 367.67126701, 0.31060611),
 (-134.99284089, 364.80563353, 0.33104816)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4961267.91145195, 1982012.08009271, 3475323.76587499)] m, obsgeovel=[(-134.99284089, 364.80563353, 0.33104816),
 (-144.52193431, 361.13655472, 0.35536867)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4969891.79716081, 1960324.98562162, 3475302.56636067)] m, obsgeovel=[(-134.99284089, 364.80563353, 0.33104816),
 (-142.94049161, 361.76542035, 0.3513318 )] m / s)>. Angular separation can dep


Day 5
Reward: 0.0
Current day: 6
Requested: [{'candidate_id': 'SLSN-II_6005', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'SLSN-II_6744', 'exptime_s': 180, 'status': 'EXECUTED'}]
Executed: [{'candidate_id': 'SLSN-II_6744', 'exptime_s': 180, 'scheduled_time': 5.192578306732078, 'n_sedm_points': 3, 'n_sedm_detections': 2, 'detected_filters': ['i', 'r'], 'has_sedm_detection': True}]
Executed candidate details:
  SLSN-II_6744: exptime=180 s, scheduled_time=5.1926, n_sedm_points=3, n_sedm_detections=2, detected_filters=['i', 'r'], has_sedm_detection=True


 (4963940.09659318, 1975324.88404249, 3475315.34862793)] m, obsgeovel=[(-134.50018908, 364.98755624, 0.32972575),
 (-144.03422742, 361.33134622, 0.35406202)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4972534.69466804, 1953626.1663368 , 3475294.22711497)] m, obsgeovel=[(-134.50018908, 364.98755624, 0.32972575),
 (-142.45193714, 361.95807614, 0.35002247)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4919547.63746537, 2083243.02006768, 3475424.58047072)] m, obsgeovel=[(-142.45193714, 361.95807614, 0.35002247),
 (-151.9037163 , 358.094187  , 0.3741566 )] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4928614.82826541, 2061737.49835932, 3475402.25225127)] m, obsgeovel=[(-142.45193714, 361.95807614, 0.35002247),
 (-150.33551405, 358.75537904, 0.3701517 )] m / s)>. Angular separation can dep


Day 6
Reward: 0.0
Current day: 7
Requested: [{'candidate_id': 'Ia_445', 'exptime_s': 180, 'status': 'PENDING'}, {'candidate_id': 'SLSN-I_4108', 'exptime_s': 120, 'status': 'EXECUTED'}]
Executed: [{'candidate_id': 'SLSN-I_4108', 'exptime_s': 120, 'scheduled_time': 6.085467018135306, 'n_sedm_points': 3, 'n_sedm_detections': 0, 'detected_filters': [], 'has_sedm_detection': False}]
Executed candidate details:
  SLSN-I_4108: exptime=120 s, scheduled_time=6.0855, n_sedm_points=3, n_sedm_detections=0, detected_filters=[], has_sedm_detection=False
Episode ended.


# Fixed KN to follow up for 7 days to analyse reward function

In [1]:
#!/usr/bin/env python3
"""
kn_sedm_followup_env_scheduler_no_cadence.py

Full Gymnasium environment for kilonova follow-up with:
- variable number of candidates
- masked fixed-size observation tensor
- nightly PENDING queue
- scheduler inspired by the SEDM logic
- NO historical SEDM cadence
- exactly one executed SEDM batch per night in g, r, i
- reward from improvement in chi^2 of a linear MAGNITUDE-decay model

Important conventions
---------------------
1) Action meanings:
   0 = no request
   1 = 90 s batch total for g+r+i
   2 = 120 s batch total for g+r+i
   3 = 180 s batch total for g+r+i

2) Per-candidate scheduled time cost:
      batch_exptime_total + slew_overhead_s

   so if exptime_s = 90, total cost = 90 + 180 = 270 s

3) Observation returned by env:
   (
       gw_scalar_vec,      shape (G_scalar,)
       chirp_mass_bins,    shape (G_bins,)
       events,             shape (K_MAX, N_OBS_MAX, 5)
       cand_mask,          shape (K_MAX,)
       obs_mask,           shape (K_MAX, N_OBS_MAX)
       day                 shape (1,)
   )

4) Event features per row:
   [time_days, mag, mag_sigma, mag_limit, filter_id]

Expected files in each scenario directory
-----------------------------------------
scenario_metadata.csv
true_all.csv
KN_true.csv
observe_all.csv
KN_observe.csv
"""

from __future__ import annotations

import os
import glob
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

from sklearn.neighbors import KernelDensity
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

import astropy.units as u
from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_body
from astroplan import (
    Observer,
    FixedTarget,
    AltitudeConstraint,
    AirmassConstraint,
    MoonSeparationConstraint,
    is_observable,
)


# ============================================================
# Constants
# ============================================================
VALID_CANON_BANDS = ("g", "r", "i")

FILTER_MAP_HIST_TO_CANON = {
    "sdssg": "g",
    "sdssr": "r",
    "sdssi": "i",
    "g": "g",
    "r": "r",
    "i": "i",
}

FILTER_TO_ID = {
    "u": 0,
    "g": 1,
    "r": 2,
    "i": 3,
    "z": 4,
    "y": 5,
}

ACTION_TO_EXPTIME = {
    0: None,
    1: 90,
    2: 120,
    3: 180,
}

ALL_MODEL_FILTERS = tuple(FILTER_TO_ID.keys())


# ============================================================
# General utilities
# ============================================================
def _sigmoid(x: float) -> float:
    return float(1.0 / (1.0 + np.exp(-x)))


def safe_read_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path)


def build_gw_scalar_vector(meta_row: pd.Series) -> np.ndarray:
    vals = [
        float(meta_row.get("chirp_mass", np.nan)),
        float(meta_row.get("HasNS", np.nan)),
        float(meta_row.get("HasRemnant", np.nan)),
        float(meta_row.get("HasMassGap", np.nan)),
        float(meta_row.get("area90", np.nan)),
        float(meta_row.get("distmean", np.nan)),
        float(meta_row.get("diststd", np.nan)),
        float(meta_row.get("ra", np.nan)),
        float(meta_row.get("dec", np.nan)),
    ]
    vals = np.asarray(vals, dtype=np.float32)
    vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
    return vals


def build_chirp_mass_bins(meta_row: pd.Series) -> np.ndarray:
    edges = meta_row.get("chirp_mass_bin_edges", None)

    if isinstance(edges, str):
        try:
            arr = np.fromstring(edges.strip("[]"), sep=",", dtype=np.float32)
            arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            return arr
        except Exception:
            pass

    return np.zeros((0,), dtype=np.float32)


def weighted_linear_fit_result(
    x: np.ndarray,
    y: np.ndarray,
    yerr: np.ndarray,
) -> Optional[Dict[str, float]]:
    """
    Fit y = a + b x with weights 1/yerr^2 and return the best-fit parameters
    plus chi^2 and reduced chi^2.

    chi^2(a,b) = sum_i [ (y_i - (a + b x_i)) / yerr_i ]^2
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    yerr = np.asarray(yerr, float)

    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(yerr) & (yerr > 0)
    x, y, yerr = x[mask], y[mask], yerr[mask]

    if len(x) < 2:
        return None

    w = 1.0 / (yerr ** 2)

    S = np.sum(w)
    Sx = np.sum(w * x)
    Sy = np.sum(w * y)
    Sxx = np.sum(w * x * x)
    Sxy = np.sum(w * x * y)

    denom = S * Sxx - Sx * Sx
    if denom <= 0:
        return None

    a = (Sxx * Sy - Sx * Sxy) / denom
    b = (S * Sxy - Sx * Sy) / denom

    yhat = a + b * x
    chi2 = np.sum(((y - yhat) / yerr) ** 2)
    dof = max(len(x) - 2, 1)
    red_chi2 = chi2 / dof

    return {
        "a": float(a),
        "b": float(b),
        "chi2": float(chi2),
        "reduced_chi2": float(red_chi2),
        "n_points": int(len(x)),
    }


def compute_mag_fit_stats_by_filter(
    df: pd.DataFrame,
    allowed_filters: Tuple[str, ...] = ALL_MODEL_FILTERS,
    min_points_per_filter: int = 2,
) -> Dict[str, Dict[str, float]]:
    """
    For each filter, return:
      - a
      - b
      - chi2
      - reduced_chi2
      - n_points
    """
    out: Dict[str, Dict[str, float]] = {}

    if df is None or df.empty:
        return out

    dff = df.copy()
    dff["filter"] = dff["filter"].astype(str).str.strip().str.lower()
    dff["time"] = pd.to_numeric(dff["time"], errors="coerce")
    dff["mag"] = pd.to_numeric(dff["mag"], errors="coerce")
    dff["magerr"] = pd.to_numeric(dff["magerr"], errors="coerce")

    dff = dff[
        np.isfinite(dff["time"])
        & np.isfinite(dff["mag"])
        & np.isfinite(dff["magerr"])
        & (dff["magerr"] > 0)
    ]
    dff = dff[dff["filter"].isin(allowed_filters)]

    if dff.empty:
        return out

    for band in allowed_filters:
        sub = dff[dff["filter"] == band].sort_values("time")
        if len(sub) < min_points_per_filter:
            continue

        x = sub["time"].to_numpy(float)
        y = sub["mag"].to_numpy(float)
        yerr = sub["magerr"].to_numpy(float)

        fit = weighted_linear_fit_result(x, y, yerr)
        if fit is not None:
            out[str(band)] = fit

    return out


def compute_average_filter_improvement(
    before_map: Dict[str, Dict[str, float]],
    after_map: Dict[str, Dict[str, float]],
    min_filters_required: int = 2,
    use_reduced_chi2: bool = True,
) -> Tuple[float, List[str], Dict[str, float]]:
    """
    Improvement is defined as chi2_before - chi2_after using the minimum chi2
    from the best-fit line in each filter.
    """
    valid_filters = sorted(set(before_map.keys()) & set(after_map.keys()))
    per_filter_improvement: Dict[str, float] = {}

    key = "reduced_chi2" if use_reduced_chi2 else "chi2"

    for f in valid_filters:
        b = float(before_map[f][key])
        a = float(after_map[f][key])
        per_filter_improvement[f] = b - a

    if len(valid_filters) < min_filters_required:
        return 0.0, valid_filters, per_filter_improvement

    avg_improvement = float(np.mean([per_filter_improvement[f] for f in valid_filters]))
    return avg_improvement, valid_filters, per_filter_improvement


# ============================================================
# Truth LC handling
# ============================================================
@dataclass
class TruthLC:
    t_by_band: Dict[str, np.ndarray]
    m_by_band: Dict[str, np.ndarray]

    def mag(self, t: float, band: str) -> float:
        if band not in self.t_by_band:
            return np.nan
        tt = self.t_by_band[band]
        mm = self.m_by_band[band]
        if tt.size == 0:
            return np.nan
        if t <= tt[0]:
            return float(mm[0])
        if t >= tt[-1]:
            return float(mm[-1])
        return float(np.interp(t, tt, mm))


def truth_df_to_truthlc(
    truth_df: pd.DataFrame,
    keep_bands: Tuple[str, ...] = VALID_CANON_BANDS,
) -> TruthLC:
    df = truth_df.copy()
    df["filter"] = df["filter"].astype(str).str.strip().str.lower()
    df["time"] = pd.to_numeric(df["time"], errors="coerce")
    df["mag"] = pd.to_numeric(df["mag"], errors="coerce")
    df = df[np.isfinite(df["time"]) & np.isfinite(df["mag"])]
    df = df[df["filter"].isin(keep_bands)]

    t_by_band = {}
    m_by_band = {}
    for band, gb in df.groupby("filter"):
        tt = gb["time"].to_numpy(float)
        mm = gb["mag"].to_numpy(float)
        order = np.argsort(tt)
        t_by_band[band] = tt[order]
        m_by_band[band] = mm[order]

    return TruthLC(t_by_band=t_by_band, m_by_band=m_by_band)


# ============================================================
# SEDM historical models (conditions + magerr only, no cadence)
# ============================================================
@dataclass
class ConditionModel:
    kdes: Dict[Tuple[int, str], KernelDensity]
    scalers: Dict[Tuple[int, str], StandardScaler]
    skynoise_max: float = 100.0

    def sample(self, exptime_s: int, band: str, rng: np.random.Generator) -> Tuple[float, float]:
        key = (int(exptime_s), str(band))
        if key not in self.kdes:
            raise KeyError(f"No KDE for exptime_s={exptime_s}, band={band}")

        kde = self.kdes[key]
        scaler = self.scalers[key]

        seed = int(rng.integers(0, 2**31 - 1))
        z_scaled = kde.sample(1, random_state=seed).reshape(1, -1)
        z = scaler.inverse_transform(z_scaled).reshape(-1)

        skynoise = float(z[0])
        mlim = float(z[1])

        if not np.isfinite(skynoise) or skynoise <= 0:
            skynoise = 1.0
        skynoise = float(np.clip(skynoise, 1e-6, self.skynoise_max))

        if not np.isfinite(mlim):
            mlim = 20.0

        return skynoise, mlim


def fit_condition_kdes_from_historical(
    hist_csv: str,
    exptimes: Tuple[int, ...] = (90, 120, 180),
    bands: Tuple[str, ...] = VALID_CANON_BANDS,
    skynoise_max: float = 100.0,
    skynoise_jitter_frac: float = 0.02,
    bw_min: float = 0.05,
    bw_max: float = 1.50,
    n_grid: int = 30,
    min_rows: int = 100,
) -> ConditionModel:
    df = pd.read_csv(hist_csv, usecols=["exptime_s", "filter", "skynoise", "limiting_mag"])
    df["exptime_s"] = pd.to_numeric(df["exptime_s"], errors="coerce")
    df["skynoise"] = pd.to_numeric(df["skynoise"], errors="coerce")
    df["limiting_mag"] = pd.to_numeric(df["limiting_mag"], errors="coerce")
    df["filter"] = df["filter"].astype(str).str.strip().str.lower()
    df["filter"] = df["filter"].map(FILTER_MAP_HIST_TO_CANON)

    df = df[np.isfinite(df["exptime_s"]) & np.isfinite(df["skynoise"]) & np.isfinite(df["limiting_mag"])]
    df = df.dropna(subset=["filter"])
    df = df[df["exptime_s"].isin(exptimes)]
    df = df[df["filter"].isin(bands)]
    df = df[(df["skynoise"] > 0) & (df["skynoise"] <= skynoise_max)]

    rng = np.random.default_rng(0)
    kdes = {}
    scalers = {}

    bw_grid = np.logspace(np.log10(bw_min), np.log10(bw_max), n_grid)

    for expt in exptimes:
        for b in bands:
            sub = df[(df["exptime_s"] == expt) & (df["filter"] == b)]
            if len(sub) < min_rows:
                continue

            skynoise = sub["skynoise"].to_numpy(float)
            if skynoise_jitter_frac > 0:
                skynoise = skynoise * (1.0 + rng.normal(0.0, skynoise_jitter_frac, size=skynoise.shape))
            mlim = sub["limiting_mag"].to_numpy(float)

            X = np.column_stack([skynoise, mlim])
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X)

            grid = GridSearchCV(
                KernelDensity(kernel="gaussian"),
                {"bandwidth": bw_grid},
                cv=5,
                n_jobs=-1,
            )
            grid.fit(Xs)

            kdes[(int(expt), str(b))] = grid.best_estimator_
            scalers[(int(expt), str(b))] = scaler

    return ConditionModel(kdes=kdes, scalers=scalers, skynoise_max=skynoise_max)


@dataclass
class MagErrSampler:
    delta_edges: np.ndarray
    bins: Dict[Tuple[int, str], List[np.ndarray]]

    def sample(self, exptime_s: int, band: str, delta: float, rng: np.random.Generator) -> float:
        key = (int(exptime_s), str(band))
        if key not in self.bins:
            return self._fallback(delta, rng)

        edges = self.delta_edges
        j = int(np.clip(np.searchsorted(edges, delta, side="right") - 1, 0, len(edges) - 2))
        arr = self.bins[key][j]

        if arr.size == 0:
            per = self.bins[key]
            for step in range(1, len(per)):
                j1 = max(0, j - step)
                j2 = min(len(per) - 1, j + step)
                if per[j1].size:
                    arr = per[j1]
                    break
                if per[j2].size:
                    arr = per[j2]
                    break

        if arr.size == 0:
            return self._fallback(delta, rng)

        return float(rng.choice(arr))

    @staticmethod
    def _fallback(delta: float, rng: np.random.Generator) -> float:
        base = 0.03 + 0.10 * _sigmoid(delta / 0.5)
        return float(np.clip(rng.normal(base, 0.02), 0.01, 0.3))


def fit_magerr_sampler_from_historical(
    hist_csv: str,
    exptimes: Tuple[int, ...] = (90, 120, 180),
    bands: Tuple[str, ...] = VALID_CANON_BANDS,
    skynoise_max: float = 100.0,
    max_magerr: float = 0.5,
    delta_edges: Optional[np.ndarray] = None,
) -> MagErrSampler:
    if delta_edges is None:
        delta_edges = np.array(
            [-6, -4, -3, -2, -1.5, -1.0, -0.7, -0.5, -0.3, -0.2, -0.1, 0.0, 0.2, 0.5, 1.0],
            dtype=float,
        )

    df = pd.read_csv(hist_csv, usecols=["exptime_s", "filter", "mag", "magerr", "limiting_mag", "skynoise"])
    df["exptime_s"] = pd.to_numeric(df["exptime_s"], errors="coerce")
    df["mag"] = pd.to_numeric(df["mag"], errors="coerce")
    df["magerr"] = pd.to_numeric(df["magerr"], errors="coerce")
    df["limiting_mag"] = pd.to_numeric(df["limiting_mag"], errors="coerce")
    df["skynoise"] = pd.to_numeric(df["skynoise"], errors="coerce")
    df["filter"] = df["filter"].astype(str).str.strip().str.lower()
    df["filter"] = df["filter"].map(FILTER_MAP_HIST_TO_CANON)

    df = df[
        np.isfinite(df["exptime_s"])
        & np.isfinite(df["mag"])
        & np.isfinite(df["magerr"])
        & np.isfinite(df["limiting_mag"])
        & np.isfinite(df["skynoise"])
    ]
    df = df.dropna(subset=["filter"])
    df = df[df["exptime_s"].isin(exptimes)]
    df = df[df["filter"].isin(bands)]
    df = df[(df["skynoise"] > 0) & (df["skynoise"] <= skynoise_max)]
    df = df[(df["magerr"] > 0) & (df["magerr"] <= max_magerr)]

    df["delta"] = df["mag"] - df["limiting_mag"]

    edges = np.asarray(delta_edges, dtype=float)
    nb = len(edges) - 1

    bins = {}
    for expt in exptimes:
        for b in bands:
            sub = df[(df["exptime_s"] == expt) & (df["filter"] == b)]
            d = sub["delta"].to_numpy(float)
            e = sub["magerr"].to_numpy(float)

            per_bin = []
            for i in range(nb):
                m = (d >= edges[i]) & (d < edges[i + 1])
                per_bin.append(e[m])
            bins[(int(expt), str(b))] = per_bin

    return MagErrSampler(delta_edges=edges, bins=bins)


def ensure_condition_fallbacks(
    conditions: ConditionModel,
    exptimes: Tuple[int, ...] = (90, 120, 180),
    bands: Tuple[str, ...] = VALID_CANON_BANDS,
) -> None:
    have = set(conditions.kdes.keys())
    for b in bands:
        avail = sorted([e for (e, bb) in have if bb == b])
        if not avail:
            continue
        for e in exptimes:
            key = (int(e), str(b))
            if key in have:
                continue
            nearest = min(avail, key=lambda x: abs(x - e))
            conditions.kdes[key] = conditions.kdes[(nearest, b)]
            conditions.scalers[key] = conditions.scalers[(nearest, b)]


class SEDMObservationModel:
    """
    Scheduler-only SEDM generator.

    The scheduler decides when the observation happens.
    For each executed request, generate exactly one observation
    in g, r, and i at that scheduled time.

    Historical SEDM is used only for:
    - limiting_mag / skynoise sampling
    - magerr sampling
    NOT for cadence / visit timing.
    """

    def __init__(
        self,
        hist_csv: str,
        exptimes: Tuple[int, ...] = (90, 120, 180),
        bands: Tuple[str, ...] = VALID_CANON_BANDS,
        skynoise_max: float = 100.0,
        max_magerr: float = 0.5,
    ):
        self.conditions = fit_condition_kdes_from_historical(
            hist_csv,
            exptimes=exptimes,
            bands=bands,
            skynoise_max=skynoise_max,
        )
        self.magerr_sampler = fit_magerr_sampler_from_historical(
            hist_csv,
            exptimes=exptimes,
            bands=bands,
            skynoise_max=skynoise_max,
            max_magerr=max_magerr,
        )
        ensure_condition_fallbacks(self.conditions, exptimes=exptimes, bands=bands)
        self.bands = tuple(bands)

    def generate(
        self,
        source_id: str,
        truth_df: pd.DataFrame,
        exptime_s: int,
        t_start: float,
        rng: np.random.Generator,
        max_visits: int = 1,  # kept for API compatibility
    ) -> pd.DataFrame:
        truth_lc = truth_df_to_truthlc(truth_df, keep_bands=self.bands)

        rows = []
        t_obs = float(t_start)

        for band in self.bands:
            m_true = truth_lc.mag(t_obs, band)
            skynoise, mlim = self.conditions.sample(exptime_s, band, rng)

            p_det = 0.0
            if np.isfinite(m_true):
                p_det = _sigmoid((mlim - m_true) / 0.25)

            is_det = bool(rng.random() < p_det)

            mag = np.nan
            magerr = np.nan

            if is_det and np.isfinite(m_true):
                delta = float(m_true - mlim)
                magerr = self.magerr_sampler.sample(exptime_s, band, delta, rng)
                mag = float(rng.normal(m_true, magerr))

            rows.append(
                {
                    "source_id": str(source_id),
                    "time": t_obs,
                    "filter": str(band),
                    "mag": mag,
                    "magerr": magerr,
                    "limiting_mag": float(mlim),
                }
            )

        return pd.DataFrame(rows)


# ============================================================
# Scheduler data classes
# ============================================================
@dataclass
class FollowupRequest:
    request_id: str
    candidate_id: str
    slot_idx: int
    exptime_s: int
    request_day: int
    priority: float
    ra: float
    dec: float
    status: str = "PENDING"
    scheduled_time: Optional[float] = None
    executed_day: Optional[int] = None
    reason: str = ""


@dataclass
class CandidateState:
    source_id: str
    slot_idx: int
    cand_type: str
    ra: float
    dec: float
    z: float
    is_kn: bool = False
    observed_rows: List[dict] = field(default_factory=list)

    mag_fit_before: Dict[str, float] = field(default_factory=dict)
    mag_fit_after: Dict[str, float] = field(default_factory=dict)

    def add_rows(self, rows: List[dict]) -> None:
        self.observed_rows.extend(rows)

    def observed_df(self) -> pd.DataFrame:
        if not self.observed_rows:
            return pd.DataFrame(columns=["time", "filter", "mag", "magerr", "limmag", "origin"])
        return pd.DataFrame(self.observed_rows).sort_values(["time", "filter"]).reset_index(drop=True)


# ============================================================
# Scheduler
# ============================================================
class SimplifiedSEDMScheduler:
    def __init__(
        self,
        location: Optional[EarthLocation] = None,
        slew_overhead_s: float = 180.0,
        focus_time_s: float = 300.0,
        standard_time_s: float = 300.0,
        nightly_budget_hours: float = 5.0,
        altitude_min_deg: float = 15.0,
        airmass_range: Tuple[float, float] = (1.0, 2.8),
    ):
        if location is None:
            location = EarthLocation(
                lat=33.3563 * u.deg,
                lon=-116.8648 * u.deg,
                height=1706 * u.m,
            )

        self.location = location
        self.observer = Observer(location=self.location, name="Palomar", timezone="UTC")

        self.slew_overhead_s = float(slew_overhead_s)
        self.focus_time_s = float(focus_time_s)
        self.standard_time_s = float(standard_time_s)
        self.nightly_budget_s = float(nightly_budget_hours) * 3600.0

        self.altitude_min_deg = float(altitude_min_deg)
        self.airmass_range = airmass_range

    def _night_bounds(self, base_date: Time) -> Tuple[Time, Time]:
        start_time = self.observer.twilight_evening_nautical(base_date, which="next")
        end_time = self.observer.twilight_morning_astronomical(start_time, which="next")
        return start_time, end_time

    def build_target_table(self, requests: List[FollowupRequest], current_time: Time) -> pd.DataFrame:
        rows = []
        for req in requests:
            if req.status != "PENDING":
                continue

            coord = SkyCoord(ra=req.ra * u.deg, dec=req.dec * u.deg)
            fixed_object = FixedTarget(coord=coord, name=req.candidate_id)

            obs_total = req.exptime_s + self.slew_overhead_s

            rows.append(
                {
                    "request_id": req.request_id,
                    "candidate_id": req.candidate_id,
                    "slot_idx": req.slot_idx,
                    "priority": float(req.priority),
                    "objname": req.candidate_id,
                    "ra": float(req.ra),
                    "dec": float(req.dec),
                    "fixed_object": fixed_object,
                    "SkyCoords": coord,
                    "obs_seq": {
                        "ifu_exptime": 0,
                        "rc": True,
                        "rc_obs_dict": {"obs_order": "gri", "obs_exptime": req.exptime_s},
                        "total": float(obs_total),
                    },
                    "maxairmass": self.airmass_range[1],
                    "typedesig": "f",
                }
            )

        if len(rows) == 0:
            return pd.DataFrame()

        targets = pd.DataFrame(rows)
        targets = self.update_targets_coords(targets, current_time)
        return targets

    def update_targets_coords(self, target_list: pd.DataFrame, obsdatetime: Time) -> pd.DataFrame:
        if len(target_list) == 0:
            return target_list

        out = target_list.copy()

        start_alt = []
        end_alt = []
        start_airmass = []
        end_airmass = []
        set_time = []

        for row in out.itertuples():
            coord = row.SkyCoords

            start_altaz = coord.transform_to(AltAz(obstime=obsdatetime, location=self.location))
            finish = obsdatetime + TimeDelta(row.obs_seq["total"], format="sec")
            end_altaz = coord.transform_to(AltAz(obstime=finish, location=self.location))

            try:
                target_set = self.observer.target_set_time(
                    obsdatetime,
                    row.fixed_object,
                    which="next",
                    horizon=0 * u.deg,
                )
                target_set_val = float(target_set.jd)
            except Exception:
                fallback_set = obsdatetime + TimeDelta(10 * 3600, format="sec")
                target_set_val = float(fallback_set.jd)

            start_alt.append(float(start_altaz.alt.deg))
            end_alt.append(float(end_altaz.alt.deg))

            sa = float(start_altaz.secz) if np.isfinite(start_altaz.secz) else np.nan
            ea = float(end_altaz.secz) if np.isfinite(end_altaz.secz) else np.nan
            start_airmass.append(sa)
            end_airmass.append(ea)
            set_time.append(target_set_val)

        out["start_alt"] = start_alt
        out["end_alt"] = end_alt
        out["start_airmass"] = start_airmass
        out["end_airmass"] = end_airmass
        out["set_time"] = set_time

        return out

    def get_next_observable_target(
        self,
        target_list: pd.DataFrame,
        obsdatetime: Time,
        remaining_budget_s: float,
        do_airmass: bool = True,
        do_moon_sep: bool = True,
        sort_columns: Tuple[str, ...] = ("priority", "set_time"),
        sort_order: Tuple[bool, ...] = (False, False),
        update_coords: bool = True,
    ) -> Tuple[Optional[str], Optional[dict]]:
        if len(target_list) == 0:
            return None, None

        if update_coords:
            target_list = self.update_targets_coords(target_list, obsdatetime)

        target_list = target_list.sort_values(list(sort_columns), ascending=list(sort_order))

        for row in target_list.itertuples():
            total_time_s = float(row.obs_seq["total"])
            if total_time_s > remaining_budget_s:
                continue

            start = obsdatetime
            finish = start + TimeDelta(total_time_s, format="sec")

            constraints = [AltitudeConstraint(min=self.altitude_min_deg * u.deg)]

            if do_airmass:
                constraints.append(AirmassConstraint(min=self.airmass_range[0], max=row.maxairmass))

            moon = get_body("moon", start, location=self.location)
            moon_dist = moon.separation(row.SkyCoords).deg

            if do_moon_sep:
                moon_illum = self.observer.moon_illumination(start) * 100.0
                min_moon_sep = 5.0 + max(0.0, moon_illum - 75.0)
                constraints.append(MoonSeparationConstraint(min=min_moon_sep * u.deg))

            observable = is_observable(
                constraints,
                self.observer,
                row.fixed_object,
                times=[start, finish],
                time_grid_resolution=0.1 * u.hour,
            )

            if not observable:
                continue

            s_air = float(row.start_airmass) if np.isfinite(row.start_airmass) else 99.0
            e_air = float(row.end_airmass) if np.isfinite(row.end_airmass) else 99.0

            if s_air > 3.5 or e_air > 3.5:
                continue

            chosen = {
                "request_id": row.request_id,
                "candidate_id": row.candidate_id,
                "slot_idx": row.slot_idx,
                "start_time": start,
                "finish_time": finish,
                "obs_seq": row.obs_seq,
                "moon_dist": float(moon_dist),
                "start_airmass": s_air,
                "end_airmass": e_air,
                "start_alt": float(row.start_alt),
                "end_alt": float(row.end_alt),
                "duration_s": total_time_s,
            }
            return row.request_id, chosen

        return None, None

    def simulate_night(
        self,
        requests: List[FollowupRequest],
        night_date: Time,
        do_focus: bool = True,
        do_standard: bool = True,
        sort_columns: Tuple[str, ...] = ("priority", "start_alt"),
        sort_order: Tuple[bool, ...] = (False, False),
    ) -> List[dict]:
        start_time, twilight_end = self._night_bounds(night_date)

        budget_end = start_time + TimeDelta(self.nightly_budget_s, format="sec")
        end_time = min(twilight_end, budget_end)

        targets = self.build_target_table(requests, start_time)
        if len(targets) == 0:
            return []

        current_time = start_time
        executed = []

        if do_focus:
            current_time += TimeDelta(self.focus_time_s, format="sec")
            do_focus = False

        if do_standard:
            current_time += TimeDelta(self.standard_time_s, format="sec")
            do_standard = False

        while current_time < end_time and len(targets) > 0:
            remaining_budget_s = (end_time - current_time).sec
            if remaining_budget_s <= 0:
                break

            targets = self.update_targets_coords(targets, current_time)
            targets = targets.sort_values(list(sort_columns), ascending=list(sort_order))

            req_id, chosen = self.get_next_observable_target(
                targets,
                obsdatetime=current_time,
                remaining_budget_s=remaining_budget_s,
                do_airmass=True,
                do_moon_sep=True,
                sort_columns=("priority", "set_time"),
                sort_order=(False, False),
                update_coords=False,
            )

            if req_id is None:
                current_time += TimeDelta(300, format="sec")
                continue

            executed.append(chosen)
            targets = targets[targets.request_id != req_id]
            current_time += TimeDelta(chosen["duration_s"], format="sec")

        return executed


# ============================================================
# Main environment
# ============================================================
class KilonovaSEDMEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        scenario_dirs: List[str],
        hist_sedm_csv: str,
        k_max: int,
        n_obs_max: int,
        n_days: int = 7,
        initial_days_revealed: float = 2.0,
        expire_after_days: int = 7,
        non_execution_penalty: float = 0.0,
        episode_start_time: str = "2026-01-01 00:00:00",
        min_filters_for_reward: int = 1,
        reward_filters: Tuple[str, ...] = ALL_MODEL_FILTERS,
        nightly_budget_hours: float = 5.0,
        slew_overhead_s: float = 180.0,
        seed: Optional[int] = None,
    ):
        super().__init__()

        if len(scenario_dirs) == 0:
            raise ValueError("scenario_dirs is empty")

        self.scenario_dirs = scenario_dirs
        self.hist_sedm_csv = hist_sedm_csv
        self.K_MAX = int(k_max)
        self.N_OBS_MAX = int(n_obs_max)
        self.N_DAYS = int(n_days)
        self.initial_days_revealed = float(initial_days_revealed)
        self.expire_after_days = int(expire_after_days)
        self.non_execution_penalty = float(non_execution_penalty)
        self.min_filters_for_reward = int(min_filters_for_reward)
        self.reward_filters = tuple(reward_filters)

        self.nightly_budget_hours = float(nightly_budget_hours)
        self.slew_overhead_s = float(slew_overhead_s)

        self.rng = np.random.default_rng(seed)
        self.ref_time = Time(episode_start_time, scale="utc")

        self.sedm_model = SEDMObservationModel(hist_csv=self.hist_sedm_csv)
        self.scheduler = SimplifiedSEDMScheduler(
            slew_overhead_s=self.slew_overhead_s,
            nightly_budget_hours=self.nightly_budget_hours,
        )

        self.G_scalar, self.G_bins = self._infer_g_dims()

        self.observation_space = spaces.Tuple((
            spaces.Box(-np.inf, np.inf, shape=(self.G_scalar,), dtype=np.float32),
            spaces.Box(-np.inf, np.inf, shape=(self.G_bins,), dtype=np.float32),
            spaces.Box(-np.inf, np.inf, shape=(self.K_MAX, self.N_OBS_MAX, 5), dtype=np.float32),
            spaces.MultiBinary(self.K_MAX),
            spaces.MultiBinary((self.K_MAX, self.N_OBS_MAX)),
            spaces.Box(0.0, float(self.N_DAYS), shape=(1,), dtype=np.float32),
        ))

        self.action_space = spaces.MultiDiscrete([4] * self.K_MAX)

        self.current_day = 0
        self.scenario_dir: Optional[str] = None

        self.gw_scalar_vec: Optional[np.ndarray] = None
        self.chirp_mass_bins: Optional[np.ndarray] = None

        self.candidates: Dict[str, CandidateState] = {}
        self.slot_to_source: Dict[int, str] = {}
        self.source_to_slot: Dict[str, int] = {}

        self.truth_map: Dict[str, pd.DataFrame] = {}
        self.rubin_obs_map: Dict[str, pd.DataFrame] = {}

        self.pending_requests: List[FollowupRequest] = []
        self.requested_last_step: List[FollowupRequest] = []
        self.executed_last_step: List[FollowupRequest] = []

        self._obs_tensor: Optional[np.ndarray] = None
        self._obs_mask: Optional[np.ndarray] = None
        self._cand_mask: Optional[np.ndarray] = None

        self.last_rubin_reveal_time: float = 0.0
        self._request_counter: int = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        self.current_day = 0
        self.pending_requests = []
        self.requested_last_step = []
        self.executed_last_step = []
        self._request_counter = 0

        self._load_random_scenario()
        self._initialize_buffers()

        self.last_rubin_reveal_time = float(self.initial_days_revealed)
        self._reveal_rubin_data_until(self.last_rubin_reveal_time)

        obs = self._build_observation()
        info = self._build_info(initial=True)
        return obs, info

    def step(self, action):
        action = np.asarray(action, dtype=int)
        if action.shape != (self.K_MAX,):
            raise ValueError(f"Expected action shape {(self.K_MAX,)}, got {action.shape}")

        before_fit_maps = self._compute_followup_filter_fit_maps(action)
        self.requested_last_step = self._enqueue_requests_from_action(action)

        night_date = self.ref_time + TimeDelta(self.current_day * 86400, format="sec")
        self.executed_last_step = self._run_scheduler_one_night(night_date)
        self._materialize_executed_requests(self.executed_last_step)

        # Reward is based on the effect of the executed follow-up itself.
        after_fit_maps = self._compute_followup_filter_fit_maps_from_requests(self.requested_last_step)

        reward, reward_breakdown = self._compute_reward(
            requested=self.requested_last_step,
            executed=self.executed_last_step,
            before_fit_maps=before_fit_maps,
            after_fit_maps=after_fit_maps,
        )

        self.current_day += 1

        new_t_hi = self.initial_days_revealed + self.current_day
        self._reveal_rubin_data_between(self.last_rubin_reveal_time, new_t_hi)
        self.last_rubin_reveal_time = new_t_hi

        obs = self._build_observation()
        terminated = self._termination_condition()
        truncated = bool(self.current_day >= self.N_DAYS and not terminated)

        info = self._build_info(
            initial=False,
            reward_breakdown=reward_breakdown,
            requested=self.requested_last_step,
            executed=self.executed_last_step,
        )
        return obs, reward, terminated, truncated, info


    def render(self):
        print(f"Day: {self.current_day}")
        print(f"Scenario: {self.scenario_dir}")
        print(f"Candidates: {len(self.candidates)}")
        print(f"Pending requests: {sum(r.status == 'PENDING' for r in self.pending_requests)}")

    # --------------------------------------------------------
    # Scenario handling
    # --------------------------------------------------------
    def _infer_g_dims(self) -> Tuple[int, int]:
        meta_path = os.path.join(self.scenario_dirs[0], "scenario_metadata.csv")
        meta = safe_read_csv(meta_path)
        row = meta.iloc[0]

        g_scalar = len(build_gw_scalar_vector(row))
        g_bins = len(build_chirp_mass_bins(row))
        return g_scalar, g_bins

    def _load_random_scenario(self) -> None:
        self.scenario_dir = str(self.rng.choice(self.scenario_dirs))
        self._load_scenario(self.scenario_dir)

    def _load_scenario(self, scenario_dir: str) -> None:
        meta = safe_read_csv(os.path.join(scenario_dir, "scenario_metadata.csv"))
        true_all = safe_read_csv(os.path.join(scenario_dir, "true_all.csv"))
        obs_all = safe_read_csv(os.path.join(scenario_dir, "observe_all.csv"))

        row = meta.iloc[0]
        self.gw_scalar_vec = build_gw_scalar_vector(row).astype(np.float32)
        self.chirp_mass_bins = build_chirp_mass_bins(row).astype(np.float32)

        for df in [true_all, obs_all]:
            df["source_id"] = df["source_id"].astype(str)
            df["filter"] = df["filter"].astype(str).str.strip().str.lower()
            df["time"] = pd.to_numeric(df["time"], errors="coerce")
            df["ra"] = pd.to_numeric(df["ra"], errors="coerce")
            df["dec"] = pd.to_numeric(df["dec"], errors="coerce")
            df["z"] = pd.to_numeric(df["z"], errors="coerce")
            if "mag" in df.columns:
                df["mag"] = pd.to_numeric(df["mag"], errors="coerce")
            if "magerr" in df.columns:
                df["magerr"] = pd.to_numeric(df["magerr"], errors="coerce")
            if "limmag" in df.columns:
                df["limmag"] = pd.to_numeric(df["limmag"], errors="coerce")

        source_ids = sorted(true_all["source_id"].unique().tolist())
        if len(source_ids) > self.K_MAX:
            raise ValueError(f"Scenario has {len(source_ids)} candidates but K_MAX={self.K_MAX}")

        self.candidates = {}
        self.slot_to_source = {}
        self.source_to_slot = {}

        for slot_idx, sid in enumerate(source_ids):
            rows = true_all[true_all["source_id"] == sid]
            first = rows.iloc[0]
            cand_type = str(first.get("type", "unknown"))
            is_kn = cand_type.upper() == "KN"

            cand = CandidateState(
                source_id=sid,
                slot_idx=slot_idx,
                cand_type=cand_type,
                ra=float(first["ra"]),
                dec=float(first["dec"]),
                z=float(first["z"]),
                is_kn=is_kn,
            )
            self.candidates[sid] = cand
            self.slot_to_source[slot_idx] = sid
            self.source_to_slot[sid] = slot_idx

        self.truth_map = {
            sid: g.copy().sort_values(["time", "filter"]).reset_index(drop=True)
            for sid, g in true_all.groupby("source_id")
        }
        self.rubin_obs_map = {
            sid: g.copy().sort_values(["time", "filter"]).reset_index(drop=True)
            for sid, g in obs_all.groupby("source_id")
        }

    # --------------------------------------------------------
    # Observation tensor handling
    # --------------------------------------------------------
    def _initialize_buffers(self) -> None:
        self._obs_tensor = np.full((self.K_MAX, self.N_OBS_MAX, 5), np.nan, dtype=np.float32)
        self._obs_mask = np.zeros((self.K_MAX, self.N_OBS_MAX), dtype=np.int8)
        self._cand_mask = np.zeros((self.K_MAX,), dtype=np.int8)

        for slot_idx in self.slot_to_source:
            self._cand_mask[slot_idx] = 1

        for cand in self.candidates.values():
            cand.observed_rows = []
            cand.mag_fit_before = {}
            cand.mag_fit_after = {}

    def _build_observation(self):
        return (
            self.gw_scalar_vec.astype(np.float32),
            self.chirp_mass_bins.astype(np.float32),
            self._obs_tensor.astype(np.float32),
            self._cand_mask.astype(np.int8),
            self._obs_mask.astype(np.int8),
            np.asarray([self.current_day], dtype=np.float32),
        )

    def _append_observation_row(self, slot_idx: int, row: dict) -> None:
        free = np.where(self._obs_mask[slot_idx] == 0)[0]
        if len(free) == 0:
            self._obs_tensor[slot_idx, :-1] = self._obs_tensor[slot_idx, 1:]
            self._obs_mask[slot_idx, :-1] = self._obs_mask[slot_idx, 1:]
            j = self.N_OBS_MAX - 1
        else:
            j = int(free[0])

        filt = str(row["filter"]).lower()
        if filt not in FILTER_TO_ID:
            return

        arr = np.array([
            float(row.get("time", np.nan)),
            float(row.get("mag", np.nan)),
            float(row.get("magerr", np.nan)),
            float(row.get("limmag", np.nan)),
            float(FILTER_TO_ID[filt]),
        ], dtype=np.float32)

        self._obs_tensor[slot_idx, j] = arr
        self._obs_mask[slot_idx, j] = 1

    # --------------------------------------------------------
    # Rubin reveal
    # --------------------------------------------------------
    def _reveal_rubin_data_until(self, t_max: float) -> None:
        for sid, df in self.rubin_obs_map.items():
            rows = df[df["time"] <= t_max].to_dict("records")
            self._insert_new_rows(sid, rows, origin="Rubin")

    def _reveal_rubin_data_between(self, t_lo: float, t_hi: float) -> None:
        for sid, df in self.rubin_obs_map.items():
            rows = df[(df["time"] > t_lo) & (df["time"] <= t_hi)].to_dict("records")
            self._insert_new_rows(sid, rows, origin="Rubin")

    def _insert_new_rows(self, source_id: str, rows: List[dict], origin: str) -> None:
        if source_id not in self.candidates or len(rows) == 0:
            return

        cand = self.candidates[source_id]
        slot_idx = cand.slot_idx

        formatted = []
        for r in rows:
            rr = {
                "time": float(r.get("time", np.nan)),
                "filter": str(r.get("filter", "")).lower(),
                "mag": float(r.get("mag", np.nan)),
                "magerr": float(r.get("magerr", np.nan)) if "magerr" in r else np.nan,
                "limmag": float(r.get("limmag", np.nan)) if "limmag" in r else np.nan,
                "origin": origin,
            }
            formatted.append(rr)
            self._append_observation_row(slot_idx, rr)

        cand.add_rows(formatted)

    # --------------------------------------------------------
    # Requests and scheduling
    # --------------------------------------------------------
    def _candidate_has_pending_request(self, sid: str) -> bool:
        return any(r.candidate_id == sid and r.status == "PENDING" for r in self.pending_requests)

    def _compute_followup_filter_fit_maps(self, action: np.ndarray) -> Dict[str, Dict[str, Dict[str, float]]]:
        out: Dict[str, Dict[str, Dict[str, float]]] = {}
        for slot_idx, a in enumerate(action):
            sid = self.slot_to_source.get(slot_idx)
            if sid is None or int(a) == 0:
                continue
            out[sid] = compute_mag_fit_stats_by_filter(
                self.candidates[sid].observed_df(),
                allowed_filters=self.reward_filters,
                min_points_per_filter=2,
            )
        return out

    def _compute_followup_filter_fit_maps_from_requests(
        self,
        requests: List[FollowupRequest],
    ) -> Dict[str, Dict[str, Dict[str, float]]]:
        out: Dict[str, Dict[str, Dict[str, float]]] = {}
        seen = set()
        for req in requests:
            sid = req.candidate_id
            if sid in seen:
                continue
            seen.add(sid)
            out[sid] = compute_mag_fit_stats_by_filter(
                self.candidates[sid].observed_df(),
                allowed_filters=self.reward_filters,
                min_points_per_filter=2,
            )
        return out


    def _enqueue_requests_from_action(self, action: np.ndarray) -> List[FollowupRequest]:
        requests = []
        for slot_idx, a in enumerate(action):
            a = int(a)
            sid = self.slot_to_source.get(slot_idx)
            if sid is None or a == 0:
                continue

            if self._candidate_has_pending_request(sid):
                continue

            exptime = ACTION_TO_EXPTIME[a]
            cand = self.candidates[sid]

            current_fit_map = compute_mag_fit_stats_by_filter(
                cand.observed_df(),
                allowed_filters=self.reward_filters,
                min_points_per_filter=2,
            )

            if len(current_fit_map) > 0:
                mean_chi2 = float(np.mean([v["reduced_chi2"] for v in current_fit_map.values()]))
            else:
                mean_chi2 = 10.0

            priority = 1.0 / (1.0 + mean_chi2)


            req = FollowupRequest(
                request_id=f"req_{self.current_day}_{self._request_counter}_{sid}",
                candidate_id=sid,
                slot_idx=slot_idx,
                exptime_s=int(exptime),
                request_day=self.current_day,
                priority=float(priority),
                ra=cand.ra,
                dec=cand.dec,
            )
            self._request_counter += 1

            self.pending_requests.append(req)
            requests.append(req)

        return requests

    def _run_scheduler_one_night(self, night_date: Time) -> List[FollowupRequest]:
        for req in self.pending_requests:
            if req.status == "PENDING" and (self.current_day - req.request_day) > self.expire_after_days:
                req.status = "EXPIRED"
                req.reason = "expired"

        pending = [r for r in self.pending_requests if r.status == "PENDING"]
        if len(pending) == 0:
            return []

        executed_sched = self.scheduler.simulate_night(
            requests=pending,
            night_date=night_date,
            do_focus=True,
            do_standard=True,
            sort_columns=("priority", "start_alt"),
            sort_order=(False, False),
        )

        executed_by_request_id = {e["request_id"]: e for e in executed_sched}
        executed_requests = []

        for req in pending:
            if req.request_id in executed_by_request_id:
                entry = executed_by_request_id[req.request_id]
                abs_start = entry["start_time"]
                rel_days = (abs_start - self.ref_time).sec / 86400.0

                req.status = "EXECUTED"
                req.executed_day = self.current_day
                req.scheduled_time = float(rel_days)
                req.reason = "scheduled"
                executed_requests.append(req)

        return executed_requests

    # --------------------------------------------------------
    # SEDM materialization
    # --------------------------------------------------------
    def _materialize_executed_requests(self, executed_requests: List[FollowupRequest]) -> None:
        for req in executed_requests:
            sid = req.candidate_id
            truth_df = self.truth_map[sid]

            sedm_df = self.sedm_model.generate(
                source_id=sid,
                truth_df=truth_df,
                exptime_s=req.exptime_s,
                t_start=float(req.scheduled_time),
                rng=self.rng,
                max_visits=1,
            )

            rows = []
            for _, r in sedm_df.iterrows():
                rows.append({
                    "time": float(r["time"]),
                    "filter": str(r["filter"]).lower(),
                    "mag": float(r["mag"]) if np.isfinite(r["mag"]) else np.nan,
                    "magerr": float(r["magerr"]) if np.isfinite(r["magerr"]) else np.nan,
                    "limmag": float(r["limiting_mag"]) if np.isfinite(r["limiting_mag"]) else np.nan,
                    "origin": "SEDM",
                })

            self._insert_new_rows(sid, rows, origin="SEDM")

    # --------------------------------------------------------
    # Reward
    # --------------------------------------------------------
    def _compute_reward(
        self,
        requested: List[FollowupRequest],
        executed: List[FollowupRequest],
        before_fit_maps: Dict[str, Dict[str, Dict[str, float]]],
        after_fit_maps: Dict[str, Dict[str, Dict[str, float]]],
    ) -> Tuple[float, Dict[str, Any]]:
        reward = 0.0
        executed_ids = set(r.candidate_id for r in executed)
        per_candidate = {}
        seen = set()

        for req in requested:
            sid = req.candidate_id
            if sid in seen:
                continue
            seen.add(sid)

            before_map = before_fit_maps.get(sid, {})
            after_map = after_fit_maps.get(sid, {})

            avg_improvement, valid_filters, per_filter_improvement = compute_average_filter_improvement(
                before_map=before_map,
                after_map=after_map,
                min_filters_required=self.min_filters_for_reward,
                use_reduced_chi2=True,
            )

            contrib = float(avg_improvement)
            if sid not in executed_ids:
                contrib -= self.non_execution_penalty

            reward += contrib
            per_candidate[sid] = {
                "fit_before": before_map,
                "fit_after": after_map,
                "valid_filters_used": valid_filters,
                "n_valid_filters": len(valid_filters),
                "avg_improvement": avg_improvement,
                "per_filter_improvement": per_filter_improvement,
                "executed": sid in executed_ids,
                "reward_contribution": contrib,
            }

            self.candidates[sid].mag_fit_before = {
                f: {"reduced_chi2": before_map[f]["reduced_chi2"], "a": before_map[f]["a"], "b": before_map[f]["b"]}
                for f in before_map
            }
            self.candidates[sid].mag_fit_after = {
                f: {"reduced_chi2": after_map[f]["reduced_chi2"], "a": after_map[f]["a"], "b": after_map[f]["b"]}
                for f in after_map
            }

        return float(reward), {
            "requested_ids": [r.candidate_id for r in requested],
            "executed_ids": list(executed_ids),
            "per_candidate": per_candidate,
        }


    # --------------------------------------------------------
    # Termination
    # --------------------------------------------------------
    def _termination_condition(self) -> bool:
        scores = []
        for sid, cand in self.candidates.items():
            fit_map = compute_mag_fit_stats_by_filter(
                cand.observed_df(),
                allowed_filters=self.reward_filters,
                min_points_per_filter=2,
            )
            if len(fit_map) >= self.min_filters_for_reward:
                mean_chi2 = float(np.mean([v["reduced_chi2"] for v in fit_map.values()]))
                scores.append((sid, mean_chi2, cand.is_kn))

        if len(scores) == 0:
            return False

        scores = sorted(scores, key=lambda x: x[1])
        best_sid, best_mean_chi2, best_is_kn = scores[0]

        if not best_is_kn:
            return False

        if len(scores) == 1:
            return True

        second_mean_chi2 = scores[1][1]
        return (second_mean_chi2 - best_mean_chi2) > 0.5


    # --------------------------------------------------------
    # Info
    # --------------------------------------------------------
    def _build_info(
        self,
        initial: bool = False,
        reward_breakdown: Optional[dict] = None,
        requested: Optional[List[FollowupRequest]] = None,
        executed: Optional[List[FollowupRequest]] = None,
    ) -> Dict[str, Any]:

        info = {
            "current_day": self.current_day,
            "n_candidates": len(self.candidates),
            "n_pending_requests": sum(r.status == "PENDING" for r in self.pending_requests),
        }

        if initial:
            return info

        requested = requested or []
        executed = executed or []

        info["requested"] = [
            {
                "candidate_id": r.candidate_id,
                "exptime_s": r.exptime_s,
                "status": r.status,
            }
            for r in requested
        ]

        executed_summary = []
        for r in executed:
            sid = r.candidate_id
            cand_df = self.candidates[sid].observed_df()

            sedm_df = cand_df[cand_df["origin"] == "SEDM"].copy()
            sedm_df["mag"] = pd.to_numeric(sedm_df["mag"], errors="coerce")

            det_df = sedm_df[np.isfinite(sedm_df["mag"])].copy()

            executed_summary.append(
                {
                    "candidate_id": sid,
                    "exptime_s": r.exptime_s,
                    "scheduled_time": r.scheduled_time,
                    "n_sedm_points": int(len(sedm_df)),
                    "n_sedm_detections": int(len(det_df)),
                    "detected_filters": sorted(det_df["filter"].astype(str).unique().tolist()),
                    "has_sedm_detection": bool(len(det_df) > 0),
                }
            )

        info["executed"] = executed_summary
        info["reward"] = reward_breakdown or {}

        return info
# ============================================================
# Exact KN matching helpers
# ============================================================
def _is_exact_kn_sid(sid: str, target_id: str) -> bool:
    sid_str = str(sid).strip()
    return sid_str in {f"KN_{target_id}", target_id}


def find_scenario_with_target(scenario_dirs: List[str], target_id: str = "8026") -> List[Tuple[str, pd.DataFrame]]:
    matches = []

    for sdir in scenario_dirs:
        true_all_path = os.path.join(sdir, "true_all.csv")
        if not os.path.exists(true_all_path):
            continue

        df = pd.read_csv(true_all_path, usecols=["source_id", "type"])
        df["source_id"] = df["source_id"].astype(str).str.strip()
        df["type"] = df["type"].astype(str).str.strip()

        mask = (df["type"].str.upper() == "KN") & df["source_id"].apply(lambda x: _is_exact_kn_sid(x, target_id))

        if mask.any():
            matches.append((sdir, df.loc[mask].copy()))

    return matches

# ============================================================
# Example usage
# ============================================================
if __name__ == "__main__":
    SCENARIO_ROOT = "/projects/bdws/npletskova1/git/Pythia/pythia/train_data_gen/BNS_O5a_combined"
    HIST_SEDM_CSV = "/projects/bdws/npletskova1/git/Pythia/pythia/train_data_gen/ZTF/all_sedm_phot_20260220_with_skynoise.csv"
    TARGET_KN_ID = "8026"

    scenario_dirs = sorted(glob.glob(os.path.join(SCENARIO_ROOT, "scenario_*")))
    print(f"Found {len(scenario_dirs)} scenarios")

    matches = find_scenario_with_target(scenario_dirs, TARGET_KN_ID)

    if len(matches) == 0:
        raise RuntimeError(f"Exact KN_{TARGET_KN_ID} was not found in any scenario.")

    print(f"\nFound exact KN_{TARGET_KN_ID} in {len(matches)} scenario(s):")
    for sdir, rows in matches:
        print(f"  {sdir}")
        print(rows[["source_id", "type"]].drop_duplicates().to_string(index=False))

    forced_scenario_dir = matches[0][0]
    print(f"\nUsing scenario: {forced_scenario_dir}")

    env = KilonovaSEDMEnv(
        scenario_dirs=[forced_scenario_dir],
        hist_sedm_csv=HIST_SEDM_CSV,
        k_max=7335,
        n_obs_max=128,
        n_days=7,
        initial_days_revealed=1.0,
        expire_after_days=6,
        non_execution_penalty=0,
        episode_start_time="2026-01-01 00:00:00",
        min_filters_for_reward=1,
        reward_filters=("g", "r", "i", "u", "z", "y"),
        nightly_budget_hours=5.0,
        slew_overhead_s=180.0,
        seed=22,
    )

    obs, info = env.reset()
    print("Reset info:", info)
    print("Loaded scenario:", env.scenario_dir)

    chosen_kn_idxs = []
    for idx, sid in env.slot_to_source.items():
        cand = env.candidates[sid]
        if cand.is_kn and _is_exact_kn_sid(sid, TARGET_KN_ID):
            chosen_kn_idxs.append(idx)

    if len(chosen_kn_idxs) == 0:
        print("\nAvailable source_ids in this scenario:")
        for idx, sid in list(env.slot_to_source.items())[:100]:
            cand = env.candidates[sid]
            print(f"  slot={idx}, source_id={sid}, type={cand.cand_type}, is_kn={cand.is_kn}")
        raise RuntimeError(f"Exact KN_{TARGET_KN_ID} not found in loaded scenario.")

    print(f"\nFollowing KN(s) from this scenario:")
    for idx in chosen_kn_idxs:
        sid = env.slot_to_source[idx]
        cand = env.candidates[sid]
        print(
            f"  slot={idx}, candidate_id={sid}, "
            f"type={cand.cand_type}, is_kn={cand.is_kn}"
        )

    OTHER_CANDIDATES_TO_FOLLOW = 10

    other_idxs = []
    for idx, sid in env.slot_to_source.items():
        if idx in chosen_kn_idxs:
            continue
        other_idxs.append(idx)

    if len(other_idxs) < OTHER_CANDIDATES_TO_FOLLOW:
        print(
            f"\nWarning: only {len(other_idxs)} other candidates available, "
            f"using all of them."
        )
        OTHER_CANDIDATES_TO_FOLLOW = len(other_idxs)

    chosen_other_idxs = env.rng.choice(
        other_idxs,
        size=OTHER_CANDIDATES_TO_FOLLOW,
        replace=False
    ).tolist() if OTHER_CANDIDATES_TO_FOLLOW > 0 else []

    tracked_idxs = chosen_kn_idxs + chosen_other_idxs
    tracked_sids = [env.slot_to_source[idx] for idx in tracked_idxs]

    print("\nTracking exact KN + 10 other candidates from this scenario:")
    for idx in tracked_idxs:
        sid = env.slot_to_source[idx]
        cand = env.candidates[sid]
        print(
            f"  slot={idx}, candidate_id={sid}, "
            f"type={cand.cand_type}, is_kn={cand.is_kn}"
        )

    for day in range(env.N_DAYS):
        action = np.zeros(env.K_MAX, dtype=int)

        for idx in tracked_idxs:
            if idx in chosen_kn_idxs:
                action[idx] = 3
            else:
                action[idx] = 2

        obs, reward, terminated, truncated, info = env.step(action)

        print(f"\n================ Day {day} ================")
        print("Reward:", reward)
        print("Current day:", info["current_day"])
        print("Requested:", info["requested"])
        print("Executed:", info["executed"])

        if len(info["executed"]) > 0:
            print("Executed candidate details:")
            for ex in info["executed"]:
                print(
                    f"  {ex['candidate_id']}: "
                    f"exptime={ex['exptime_s']} s, "
                    f"scheduled_time={ex['scheduled_time']:.4f}, "
                    f"n_sedm_points={ex['n_sedm_points']}, "
                    f"n_sedm_detections={ex['n_sedm_detections']}, "
                    f"detected_filters={ex['detected_filters']}, "
                    f"has_sedm_detection={ex['has_sedm_detection']}"
                )

        print("\nTracked candidate history:")

        for idx in tracked_idxs:
            sid = env.slot_to_source[idx]
            cand = env.candidates[sid]
            df = cand.observed_df().copy()

            rubin_df = df[df["origin"] == "Rubin"].copy()
            sedm_df = df[df["origin"] == "SEDM"].copy()

            rubin_df["mag"] = pd.to_numeric(rubin_df["mag"], errors="coerce")
            sedm_df["mag"] = pd.to_numeric(sedm_df["mag"], errors="coerce")

            n_rubin_rows = len(rubin_df)
            n_sedm_rows = len(sedm_df)

            n_rubin_det = int(np.isfinite(rubin_df["mag"]).sum()) if n_rubin_rows > 0 else 0
            n_sedm_det = int(np.isfinite(sedm_df["mag"]).sum()) if n_sedm_rows > 0 else 0

            print(
                f"  {sid} ({cand.cand_type}): "
                f"Rubin rows={n_rubin_rows}, Rubin detections={n_rubin_det}, "
                f"SEDM rows={n_sedm_rows}, SEDM detections={n_sedm_det}"
            )

        reward_pc = info.get("reward", {}).get("per_candidate", {})

        print("\nPer-candidate reward contribution:")
        for idx in tracked_idxs:
            sid = env.slot_to_source[idx]
            cand = env.candidates[sid]

            if sid in reward_pc:
                rc = reward_pc[sid]
                print(
                    f"  {sid} ({cand.cand_type}): "
                    f"executed={rc['executed']}, "
                    f"avg_improvement={rc['avg_improvement']:.4f}, "
                    f"n_valid_filters={rc['n_valid_filters']}, "
                    f"reward_contribution={rc['reward_contribution']:.4f}"
                )
            else:
                print(f"  {sid} ({cand.cand_type}): no reward entry this step")

        executed_ids = {ex["candidate_id"] for ex in info["executed"]}
        n_exec_tracked = sum(1 for sid in tracked_sids if sid in executed_ids)
        print(f"\nTracked executed tonight: {n_exec_tracked}/{len(tracked_sids)}")

        print(f"\nKN_{TARGET_KN_ID} status:")
        for idx in chosen_kn_idxs:
            sid = env.slot_to_source[idx]
            cand = env.candidates[sid]
            df = cand.observed_df().copy()

            rubin_df = df[df["origin"] == "Rubin"].copy()
            sedm_df = df[df["origin"] == "SEDM"].copy()

            rubin_df["mag"] = pd.to_numeric(rubin_df["mag"], errors="coerce")
            sedm_df["mag"] = pd.to_numeric(sedm_df["mag"], errors="coerce")

            n_total = len(df)
            n_rubin = len(rubin_df)
            n_sedm = len(sedm_df)
            n_sedm_det = int(np.isfinite(sedm_df["mag"]).sum()) if n_sedm > 0 else 0

            print(
                f"  {sid}: total_obs={n_total}, "
                f"Rubin_obs={n_rubin}, "
                f"SEDM_obs={n_sedm}, "
                f"SEDM_detections={n_sedm_det}"
            )

        if terminated or truncated:
            print("\nEpisode ended.")
            break

Found 161 scenarios

Found exact KN_8026 in 1 scenario(s):
  /projects/bdws/npletskova1/git/Pythia/pythia/train_data_gen/BNS_O5a_combined/scenario_4174
source_id type
  KN_8026   KN

Using scenario: /projects/bdws/npletskova1/git/Pythia/pythia/train_data_gen/BNS_O5a_combined/scenario_4174
Reset info: {'current_day': 0, 'n_candidates': 5664, 'n_pending_requests': 0}
Loaded scenario: /projects/bdws/npletskova1/git/Pythia/pythia/train_data_gen/BNS_O5a_combined/scenario_4174

Following KN(s) from this scenario:
  slot=3491, candidate_id=KN_8026, type=KN, is_kn=True

Tracking exact KN + 10 other candidates from this scenario:
  slot=3491, candidate_id=KN_8026, type=KN, is_kn=True
  slot=1499, candidate_id=Ia_132, type=Ia, is_kn=False
  slot=3698, candidate_id=SLSN-II_4585, type=SLSN-II, is_kn=False
  slot=1127, candidate_id=IIn_1422, type=IIn, is_kn=False
  slot=4361, candidate_id=SLSN-II_5248, type=SLSN-II, is_kn=False
  slot=5343, candidate_id=SLSN-I_4058, type=SLSN-I, is_kn=False
  slot=

 (5173770.99345595, 1333468.1552454 , 3474821.26716067)] m, obsgeovel=[(-88.96776767, 378.67007376, 0.213723  ),
 (-97.22970948, 376.63332646, 0.23474691)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5143368.56392961, 1446130.09601298, 3474894.8204914 )] m, obsgeovel=[( -97.22970948, 376.63332646, 0.23474691),
 (-105.44512148, 374.41633926, 0.25565851)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5136992.73341934, 1468581.16370829, 3474910.28299867)] m, obsgeovel=[( -97.22970948, 376.63332646, 0.23474691),
 (-107.08227392, 373.95140488, 0.25982647)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5103643.57053592, 1580406.26264557, 3474991.33846544)] m, obsgeovel=[(-107.08227392, 373.95140488, 0.25982647),
 (-115.23666264, 371.51953778, 0.28059009)] m / s)>. Angular separation can depen


================ Day 0 ================
Reward: 0.0
Current day: 1
Requested: [{'candidate_id': 'II_1092', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'IIb_2682', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'IIn_1422', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'Ia_132', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'Ia_648', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'Ic-BL_3073', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'KN_8026', 'exptime_s': 180, 'status': 'EXECUTED'}, {'candidate_id': 'SLSN-II_4585', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'status': 'EXECUTED'}, {'candidate_id': 'SLSN-II_5248', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'SLSN-I_4058', 'exptime_s': 120, 'status': 'PENDING'}]
Executed: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'scheduled_time': 0.08590926571438712, 'n_sedm_points': 3, 'n_sedm_detections': 0, 'detecte

 (5145739.48925364, 1437692.69816752, 3474885.79191652)] m, obsgeovel=[( -96.61084589, 376.79254582, 0.23328287),
 (-104.82988883, 374.58905795, 0.2542084 )] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5113063.89645104, 1549716.47058921, 3474965.16588084)] m, obsgeovel=[(-104.82988883, 374.58905795, 0.2542084 ),
 (-112.99876485, 372.20630845, 0.27501232)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5106235.13902466, 1572033.94578179, 3474981.78855535)] m, obsgeovel=[(-104.82988883, 374.58905795, 0.2542084 ),
 (-114.62617557, 371.708346  , 0.27915764)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5070630.34711359, 1683161.42868819, 3475068.62384023)] m, obsgeovel=[(-114.62617557, 371.708346  , 0.27915764),
 (-122.72969332, 369.11199524, 0.29980249)] m / s)>. Angular separation can dep


================ Day 1 ================
Reward: 0.0
Current day: 2
Requested: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'status': 'EXECUTED'}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'status': 'EXECUTED'}]
Executed: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'scheduled_time': 1.0863904037202399, 'n_sedm_points': 6, 'n_sedm_detections': 3, 'detected_filters': ['g', 'i', 'r'], 'has_sedm_detection': True}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'scheduled_time': 1.0829181814980176, 'n_sedm_points': 6, 'n_sedm_detections': 0, 'detected_filters': [], 'has_sedm_detection': False}]
Executed candidate details:
  KN_8026: exptime=180 s, scheduled_time=1.0864, n_sedm_points=6, n_sedm_detections=3, detected_filters=['g', 'i', 'r'], has_sedm_detection=True
  SLSN-II_4620: exptime=120 s, scheduled_time=1.0829, n_sedm_points=6, n_sedm_detections=0, detected_filters=[], has_sedm_detection=False

Tracked candidate history:
  KN_8026 (KN): Rubin rows=0, Rubin detections

...


================ Day 4 ================
Reward: 1.6306346237635314
Current day: 5
Requested: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'status': 'EXECUTED'}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'status': 'EXECUTED'}]
Executed: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'scheduled_time': 4.0878925762760145, 'n_sedm_points': 15, 'n_sedm_detections': 9, 'detected_filters': ['g', 'i', 'r'], 'has_sedm_detection': True}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'scheduled_time': 4.084420354053792, 'n_sedm_points': 15, 'n_sedm_detections': 0, 'detected_filters': [], 'has_sedm_detection': False}]
Executed candidate details:
  KN_8026: exptime=180 s, scheduled_time=4.0879, n_sedm_points=15, n_sedm_detections=9, detected_filters=['g', 'i', 'r'], has_sedm_detection=True
  SLSN-II_4620: exptime=120 s, scheduled_time=4.0844, n_sedm_points=15, n_sedm_detections=0, detected_filters=[], has_sedm_detection=False

Tracked candidate history:
  KN_8026 (KN): Rubin rows=0

 (5011583.45563942, 1851334.97773876, 3475200.21291638)] m, obsgeovel=[(-126.98058862, 367.67126701, 0.31060611),
 (-134.99284089, 364.80563353, 0.33104816)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4969891.79716081, 1960324.98562162, 3475302.56636067)] m, obsgeovel=[(-134.99284089, 364.80563353, 0.33104816),
 (-142.94049161, 361.76542035, 0.3513318 )] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4961267.91145195, 1982012.08009271, 3475323.76587499)] m, obsgeovel=[(-134.99284089, 364.80563353, 0.33104816),
 (-144.52193431, 361.13655472, 0.35536867)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4916729.79236215, 2089870.18401305, 3475433.38525767)] m, obsgeovel=[(-144.52193431, 361.13655472, 0.35536867),
 (-152.38704554, 357.88877368, 0.37544942)] m / s)>. Angular separation can dep


================ Day 5 ================
Reward: -39.59360530468445
Current day: 6
Requested: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'status': 'EXECUTED'}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'status': 'EXECUTED'}]
Executed: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'scheduled_time': 5.088411640065412, 'n_sedm_points': 18, 'n_sedm_detections': 10, 'detected_filters': ['g', 'i', 'r'], 'has_sedm_detection': True}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'scheduled_time': 5.08493941784319, 'n_sedm_points': 18, 'n_sedm_detections': 0, 'detected_filters': [], 'has_sedm_detection': False}]
Executed candidate details:
  KN_8026: exptime=180 s, scheduled_time=5.0884, n_sedm_points=18, n_sedm_detections=10, detected_filters=['g', 'i', 'r'], has_sedm_detection=True
  SLSN-II_4620: exptime=120 s, scheduled_time=5.0849, n_sedm_points=18, n_sedm_detections=0, detected_filters=[], has_sedm_detection=False

Tracked candidate history:
  KN_8026 (KN): Rubin rows=2

 (4972534.69466804, 1953626.1663368 , 3475294.22711497)] m, obsgeovel=[(-134.50018908, 364.98755624, 0.32972575),
 (-142.45193714, 361.95807614, 0.35002247)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4928614.82826541, 2061737.49835932, 3475402.25225127)] m, obsgeovel=[(-142.45193714, 361.95807614, 0.35002247),
 (-150.33551405, 358.75537904, 0.3701517 )] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4919547.63746537, 2083243.02006768, 3475424.58047072)] m, obsgeovel=[(-142.45193714, 361.95807614, 0.35002247),
 (-151.9037163 , 358.094187  , 0.3741566 )] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4872805.14211443, 2190164.26338609, 3475539.81402336)] m, obsgeovel=[(-151.9037163 , 358.094187  , 0.3741566 ),
 (-159.70051084, 354.68565988, 0.39407214)] m / s)>. Angular separation can dep


================ Day 6 ================
Reward: 0.0
Current day: 7
Requested: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'status': 'EXECUTED'}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'status': 'EXECUTED'}]
Executed: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'scheduled_time': 6.088939240357528, 'n_sedm_points': 21, 'n_sedm_detections': 10, 'detected_filters': ['g', 'i', 'r'], 'has_sedm_detection': True}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'scheduled_time': 6.085467018135306, 'n_sedm_points': 21, 'n_sedm_detections': 0, 'detected_filters': [], 'has_sedm_detection': False}]
Executed candidate details:
  KN_8026: exptime=180 s, scheduled_time=6.0889, n_sedm_points=21, n_sedm_detections=10, detected_filters=['g', 'i', 'r'], has_sedm_detection=True
  SLSN-II_4620: exptime=120 s, scheduled_time=6.0855, n_sedm_points=21, n_sedm_detections=0, detected_filters=[], has_sedm_detection=False

Tracked candidate history:
  KN_8026 (KN): Rubin rows=2, Rubin detect

In [42]:
sid = "II_82"
df = env.candidates[sid].observed_df()
print(df.sort_values(["time", "origin", "filter"]))
rubin_df = df[df["origin"] == "Rubin"]
sedm_df = df[df["origin"] == "SEDM"]

print("Rubin rows:", len(rubin_df))
print("Rubin detections:", np.isfinite(rubin_df["mag"]).sum())

print("SEDM rows:", len(sedm_df))
print("SEDM detections:", np.isfinite(sedm_df["mag"]).sum())

        time filter        mag    magerr     limmag origin
0   0.000000      z  22.018176  0.116818  22.652742  Rubin
1   0.086604      g        NaN       NaN  19.199927   SEDM
2   0.086604      i        NaN       NaN  17.835500   SEDM
3   0.086604      r        NaN       NaN  17.436817   SEDM
4   1.087085      g        NaN       NaN  19.275027   SEDM
5   1.087085      i        NaN       NaN  21.261935   SEDM
6   1.087085      r        NaN       NaN  21.295391   SEDM
7   2.087576      g        NaN       NaN  19.134864   SEDM
8   2.087576      i        NaN       NaN  20.628319   SEDM
9   2.087576      r        NaN       NaN  20.210803   SEDM
10  3.088077      g        NaN       NaN  19.773539   SEDM
11  3.088077      i        NaN       NaN  20.372721   SEDM
12  3.088077      r        NaN       NaN  20.178940   SEDM
13  4.088587      g        NaN       NaN  20.002662   SEDM
14  4.088587      i        NaN       NaN  17.878280   SEDM
15  4.088587      r        NaN       NaN  18.806054   SE

In [ ]:
#!/usr/bin/env python3
"""
kn_sedm_followup_env_scheduler_no_cadence_force_exact_kn8026_plus10.py
"""

from __future__ import annotations

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import glob
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

from sklearn.neighbors import KernelDensity
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

import astropy.units as u
from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_body
from astroplan import (
    Observer,
    FixedTarget,
    AltitudeConstraint,
    AirmassConstraint,
    MoonSeparationConstraint,
    is_observable,
)

# ============================================================
# Constants
# ============================================================
VALID_CANON_BANDS = ("g", "r", "i")

FILTER_MAP_HIST_TO_CANON = {
    "sdssg": "g",
    "sdssr": "r",
    "sdssi": "i",
    "g": "g",
    "r": "r",
    "i": "i",
}

FILTER_TO_ID = {
    "u": 0,
    "g": 1,
    "r": 2,
    "i": 3,
    "z": 4,
    "y": 5,
}

ACTION_TO_EXPTIME = {
    0: None,
    1: 90,
    2: 120,
    3: 180,
}

ALL_MODEL_FILTERS = tuple(FILTER_TO_ID.keys())


# ============================================================
# General utilities
# ============================================================
def _sigmoid(x: float) -> float:
    return float(1.0 / (1.0 + np.exp(-x)))


def safe_read_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path)


def build_gw_scalar_vector(meta_row: pd.Series) -> np.ndarray:
    vals = [
        float(meta_row.get("chirp_mass", np.nan)),
        float(meta_row.get("HasNS", np.nan)),
        float(meta_row.get("HasRemnant", np.nan)),
        float(meta_row.get("HasMassGap", np.nan)),
        float(meta_row.get("area90", np.nan)),
        float(meta_row.get("distmean", np.nan)),
        float(meta_row.get("diststd", np.nan)),
        float(meta_row.get("ra", np.nan)),
        float(meta_row.get("dec", np.nan)),
    ]
    vals = np.asarray(vals, dtype=np.float32)
    vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
    return vals


def build_chirp_mass_bins(meta_row: pd.Series) -> np.ndarray:
    edges = meta_row.get("chirp_mass_bin_edges", None)

    if isinstance(edges, str):
        try:
            arr = np.fromstring(edges.strip("[]"), sep=",", dtype=np.float32)
            arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            return arr
        except Exception:
            pass

    return np.zeros((0,), dtype=np.float32)


def weighted_linear_fit_chi2(x: np.ndarray, y: np.ndarray, yerr: np.ndarray) -> Optional[float]:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    yerr = np.asarray(yerr, float)

    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(yerr) & (yerr > 0)
    x, y, yerr = x[mask], y[mask], yerr[mask]

    if len(x) < 2:
        return None

    w = 1.0 / (yerr ** 2)

    S = np.sum(w)
    Sx = np.sum(w * x)
    Sy = np.sum(w * y)
    Sxx = np.sum(w * x * x)
    Sxy = np.sum(w * x * y)

    denom = S * Sxx - Sx * Sx
    if denom <= 0:
        return None

    a = (Sxx * Sy - Sx * Sxy) / denom
    b = (S * Sxy - Sx * Sy) / denom

    yhat = a + b * x
    chi2 = np.sum(((y - yhat) / yerr) ** 2)
    dof = max(len(x) - 2, 1)
    return float(chi2 / dof)


def compute_mag_fit_chi2_by_filter(
    df: pd.DataFrame,
    allowed_filters: Tuple[str, ...] = ALL_MODEL_FILTERS,
    min_points_per_filter: int = 2,
) -> Dict[str, float]:
    out: Dict[str, float] = {}

    if df is None or df.empty:
        return out

    dff = df.copy()
    dff["filter"] = dff["filter"].astype(str).str.strip().str.lower()
    dff["time"] = pd.to_numeric(dff["time"], errors="coerce")
    dff["mag"] = pd.to_numeric(dff["mag"], errors="coerce")
    dff["magerr"] = pd.to_numeric(dff["magerr"], errors="coerce")

    dff = dff[
        np.isfinite(dff["time"])
        & np.isfinite(dff["mag"])
        & np.isfinite(dff["magerr"])
        & (dff["magerr"] > 0)
    ]
    dff = dff[dff["filter"].isin(allowed_filters)]

    if dff.empty:
        return out

    for band in allowed_filters:
        sub = dff[dff["filter"] == band].sort_values("time")
        if len(sub) < min_points_per_filter:
            continue

        x = sub["time"].to_numpy(float)
        y = sub["mag"].to_numpy(float)
        yerr = sub["magerr"].to_numpy(float)

        chi2 = weighted_linear_fit_chi2(x, y, yerr)
        if chi2 is not None and np.isfinite(chi2):
            out[str(band)] = float(chi2)

    return out


def compute_average_filter_improvement(
    before_map: Dict[str, float],
    after_map: Dict[str, float],
    min_filters_required: int = 2,
) -> Tuple[float, List[str], Dict[str, float]]:
    valid_filters = sorted(set(before_map.keys()) & set(after_map.keys()))
    per_filter_improvement: Dict[str, float] = {}

    for f in valid_filters:
        b = float(before_map[f])
        a = float(after_map[f])
        per_filter_improvement[f] = max(0.0, b - a)

    if len(valid_filters) < min_filters_required:
        return 0.0, valid_filters, per_filter_improvement

    avg_improvement = float(np.mean([per_filter_improvement[f] for f in valid_filters]))
    return avg_improvement, valid_filters, per_filter_improvement


# ============================================================
# Truth LC handling
# ============================================================
@dataclass
class TruthLC:
    t_by_band: Dict[str, np.ndarray]
    m_by_band: Dict[str, np.ndarray]

    def mag(self, t: float, band: str) -> float:
        if band not in self.t_by_band:
            return np.nan
        tt = self.t_by_band[band]
        mm = self.m_by_band[band]
        if tt.size == 0:
            return np.nan
        if t <= tt[0]:
            return float(mm[0])
        if t >= tt[-1]:
            return float(mm[-1])
        return float(np.interp(t, tt, mm))


def truth_df_to_truthlc(
    truth_df: pd.DataFrame,
    keep_bands: Tuple[str, ...] = VALID_CANON_BANDS,
) -> TruthLC:
    df = truth_df.copy()
    df["filter"] = df["filter"].astype(str).str.strip().str.lower()
    df["time"] = pd.to_numeric(df["time"], errors="coerce")
    df["mag"] = pd.to_numeric(df["mag"], errors="coerce")
    df = df[np.isfinite(df["time"]) & np.isfinite(df["mag"])]
    df = df[df["filter"].isin(keep_bands)]

    t_by_band = {}
    m_by_band = {}
    for band, gb in df.groupby("filter"):
        tt = gb["time"].to_numpy(float)
        mm = gb["mag"].to_numpy(float)
        order = np.argsort(tt)
        t_by_band[band] = tt[order]
        m_by_band[band] = mm[order]

    return TruthLC(t_by_band=t_by_band, m_by_band=m_by_band)


# ============================================================
# SEDM historical models (conditions + magerr only, no cadence)
# ============================================================
@dataclass
class ConditionModel:
    kdes: Dict[Tuple[int, str], KernelDensity]
    scalers: Dict[Tuple[int, str], StandardScaler]
    skynoise_max: float = 100.0

    def sample(self, exptime_s: int, band: str, rng: np.random.Generator) -> Tuple[float, float]:
        key = (int(exptime_s), str(band))
        if key not in self.kdes:
            raise KeyError(f"No KDE for exptime_s={exptime_s}, band={band}")

        kde = self.kdes[key]
        scaler = self.scalers[key]

        seed = int(rng.integers(0, 2**31 - 1))
        z_scaled = kde.sample(1, random_state=seed).reshape(1, -1)
        z = scaler.inverse_transform(z_scaled).reshape(-1)

        skynoise = float(z[0])
        mlim = float(z[1])

        if not np.isfinite(skynoise) or skynoise <= 0:
            skynoise = 1.0
        skynoise = float(np.clip(skynoise, 1e-6, self.skynoise_max))

        if not np.isfinite(mlim):
            mlim = 20.0

        return skynoise, mlim


def fit_condition_kdes_from_historical(
    hist_csv: str,
    exptimes: Tuple[int, ...] = (90, 120, 180),
    bands: Tuple[str, ...] = VALID_CANON_BANDS,
    skynoise_max: float = 100.0,
    skynoise_jitter_frac: float = 0.02,
    bw_min: float = 0.05,
    bw_max: float = 1.50,
    n_grid: int = 30,
    min_rows: int = 100,
) -> ConditionModel:
    df = pd.read_csv(hist_csv, usecols=["exptime_s", "filter", "skynoise", "limiting_mag"])
    df["exptime_s"] = pd.to_numeric(df["exptime_s"], errors="coerce")
    df["skynoise"] = pd.to_numeric(df["skynoise"], errors="coerce")
    df["limiting_mag"] = pd.to_numeric(df["limiting_mag"], errors="coerce")
    df["filter"] = df["filter"].astype(str).str.strip().str.lower()
    df["filter"] = df["filter"].map(FILTER_MAP_HIST_TO_CANON)

    df = df[np.isfinite(df["exptime_s"]) & np.isfinite(df["skynoise"]) & np.isfinite(df["limiting_mag"])]
    df = df.dropna(subset=["filter"])
    df = df[df["exptime_s"].isin(exptimes)]
    df = df[df["filter"].isin(bands)]
    df = df[(df["skynoise"] > 0) & (df["skynoise"] <= skynoise_max)]

    rng = np.random.default_rng(0)
    kdes = {}
    scalers = {}

    bw_grid = np.logspace(np.log10(bw_min), np.log10(bw_max), n_grid)

    for expt in exptimes:
        for b in bands:
            sub = df[(df["exptime_s"] == expt) & (df["filter"] == b)]
            if len(sub) < min_rows:
                continue

            skynoise = sub["skynoise"].to_numpy(float)
            if skynoise_jitter_frac > 0:
                skynoise = skynoise * (1.0 + rng.normal(0.0, skynoise_jitter_frac, size=skynoise.shape))
            mlim = sub["limiting_mag"].to_numpy(float)

            X = np.column_stack([skynoise, mlim])
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X)

            grid = GridSearchCV(
                KernelDensity(kernel="gaussian"),
                {"bandwidth": bw_grid},
                cv=5,
                n_jobs=1,
            )
            grid.fit(Xs)

            kdes[(int(expt), str(b))] = grid.best_estimator_
            scalers[(int(expt), str(b))] = scaler

    return ConditionModel(kdes=kdes, scalers=scalers, skynoise_max=skynoise_max)


@dataclass
class MagErrSampler:
    delta_edges: np.ndarray
    bins: Dict[Tuple[int, str], List[np.ndarray]]

    def sample(self, exptime_s: int, band: str, delta: float, rng: np.random.Generator) -> float:
        key = (int(exptime_s), str(band))
        if key not in self.bins:
            return self._fallback(delta, rng)

        edges = self.delta_edges
        j = int(np.clip(np.searchsorted(edges, delta, side="right") - 1, 0, len(edges) - 2))
        arr = self.bins[key][j]

        if arr.size == 0:
            per = self.bins[key]
            for step in range(1, len(per)):
                j1 = max(0, j - step)
                j2 = min(len(per) - 1, j + step)
                if per[j1].size:
                    arr = per[j1]
                    break
                if per[j2].size:
                    arr = per[j2]
                    break

        if arr.size == 0:
            return self._fallback(delta, rng)

        return float(rng.choice(arr))

    @staticmethod
    def _fallback(delta: float, rng: np.random.Generator) -> float:
        base = 0.03 + 0.10 * _sigmoid(delta / 0.5)
        return float(np.clip(rng.normal(base, 0.02), 0.01, 0.3))


def fit_magerr_sampler_from_historical(
    hist_csv: str,
    exptimes: Tuple[int, ...] = (90, 120, 180),
    bands: Tuple[str, ...] = VALID_CANON_BANDS,
    skynoise_max: float = 100.0,
    max_magerr: float = 0.5,
    delta_edges: Optional[np.ndarray] = None,
) -> MagErrSampler:
    if delta_edges is None:
        delta_edges = np.array(
            [-6, -4, -3, -2, -1.5, -1.0, -0.7, -0.5, -0.3, -0.2, -0.1, 0.0, 0.2, 0.5, 1.0],
            dtype=float,
        )

    df = pd.read_csv(hist_csv, usecols=["exptime_s", "filter", "mag", "magerr", "limiting_mag", "skynoise"])
    df["exptime_s"] = pd.to_numeric(df["exptime_s"], errors="coerce")
    df["mag"] = pd.to_numeric(df["mag"], errors="coerce")
    df["magerr"] = pd.to_numeric(df["magerr"], errors="coerce")
    df["limiting_mag"] = pd.to_numeric(df["limiting_mag"], errors="coerce")
    df["skynoise"] = pd.to_numeric(df["skynoise"], errors="coerce")
    df["filter"] = df["filter"].astype(str).str.strip().str.lower()
    df["filter"] = df["filter"].map(FILTER_MAP_HIST_TO_CANON)

    df = df[
        np.isfinite(df["exptime_s"])
        & np.isfinite(df["mag"])
        & np.isfinite(df["magerr"])
        & np.isfinite(df["limiting_mag"])
        & np.isfinite(df["skynoise"])
    ]
    df = df.dropna(subset=["filter"])
    df = df[df["exptime_s"].isin(exptimes)]
    df = df[df["filter"].isin(bands)]
    df = df[(df["skynoise"] > 0) & (df["skynoise"] <= skynoise_max)]
    df = df[(df["magerr"] > 0) & (df["magerr"] <= max_magerr)]

    df["delta"] = df["mag"] - df["limiting_mag"]

    edges = np.asarray(delta_edges, dtype=float)
    nb = len(edges) - 1

    bins = {}
    for expt in exptimes:
        for b in bands:
            sub = df[(df["exptime_s"] == expt) & (df["filter"] == b)]
            d = sub["delta"].to_numpy(float)
            e = sub["magerr"].to_numpy(float)

            per_bin = []
            for i in range(nb):
                m = (d >= edges[i]) & (d < edges[i + 1])
                per_bin.append(e[m])
            bins[(int(expt), str(b))] = per_bin

    return MagErrSampler(delta_edges=edges, bins=bins)


def ensure_condition_fallbacks(
    conditions: ConditionModel,
    exptimes: Tuple[int, ...] = (90, 120, 180),
    bands: Tuple[str, ...] = VALID_CANON_BANDS,
) -> None:
    have = set(conditions.kdes.keys())
    for b in bands:
        avail = sorted([e for (e, bb) in have if bb == b])
        if not avail:
            continue
        for e in exptimes:
            key = (int(e), str(b))
            if key in have:
                continue
            nearest = min(avail, key=lambda x: abs(x - e))
            conditions.kdes[key] = conditions.kdes[(nearest, b)]
            conditions.scalers[key] = conditions.scalers[(nearest, b)]


class SEDMObservationModel:
    def __init__(
        self,
        hist_csv: str,
        exptimes: Tuple[int, ...] = (90, 120, 180),
        bands: Tuple[str, ...] = VALID_CANON_BANDS,
        skynoise_max: float = 100.0,
        max_magerr: float = 0.5,
    ):
        self.conditions = fit_condition_kdes_from_historical(
            hist_csv,
            exptimes=exptimes,
            bands=bands,
            skynoise_max=skynoise_max,
        )
        self.magerr_sampler = fit_magerr_sampler_from_historical(
            hist_csv,
            exptimes=exptimes,
            bands=bands,
            skynoise_max=skynoise_max,
            max_magerr=max_magerr,
        )
        ensure_condition_fallbacks(self.conditions, exptimes=exptimes, bands=bands)
        self.bands = tuple(bands)

    def generate(
        self,
        source_id: str,
        truth_df: pd.DataFrame,
        exptime_s: int,
        t_start: float,
        rng: np.random.Generator,
        max_visits: int = 1,
    ) -> pd.DataFrame:
        truth_lc = truth_df_to_truthlc(truth_df, keep_bands=self.bands)

        rows = []
        t_obs = float(t_start)

        for band in self.bands:
            m_true = truth_lc.mag(t_obs, band)
            skynoise, mlim = self.conditions.sample(exptime_s, band, rng)

            p_det = 0.0
            if np.isfinite(m_true):
                p_det = _sigmoid((mlim - m_true) / 0.25)

            is_det = bool(rng.random() < p_det)

            mag = np.nan
            magerr = np.nan

            if is_det and np.isfinite(m_true):
                delta = float(m_true - mlim)
                magerr = self.magerr_sampler.sample(exptime_s, band, delta, rng)
                mag = float(rng.normal(m_true, magerr))

            rows.append(
                {
                    "source_id": str(source_id),
                    "time": t_obs,
                    "filter": str(band),
                    "mag": mag,
                    "magerr": magerr,
                    "limiting_mag": float(mlim),
                }
            )

        return pd.DataFrame(rows)


# ============================================================
# Scheduler data classes
# ============================================================
@dataclass
class FollowupRequest:
    request_id: str
    candidate_id: str
    slot_idx: int
    exptime_s: int
    request_day: int
    priority: float
    ra: float
    dec: float
    status: str = "PENDING"
    scheduled_time: Optional[float] = None
    executed_day: Optional[int] = None
    reason: str = ""


@dataclass
class CandidateState:
    source_id: str
    slot_idx: int
    cand_type: str
    ra: float
    dec: float
    z: float
    is_kn: bool = False
    observed_rows: List[dict] = field(default_factory=list)

    mag_fit_before: Dict[str, float] = field(default_factory=dict)
    mag_fit_after: Dict[str, float] = field(default_factory=dict)

    def add_rows(self, rows: List[dict]) -> None:
        self.observed_rows.extend(rows)

    def observed_df(self) -> pd.DataFrame:
        if not self.observed_rows:
            return pd.DataFrame(columns=["time", "filter", "mag", "magerr", "limmag", "origin"])
        return pd.DataFrame(self.observed_rows).sort_values(["time", "filter"]).reset_index(drop=True)


# ============================================================
# Scheduler
# ============================================================
class SimplifiedSEDMScheduler:
    def __init__(
        self,
        location: Optional[EarthLocation] = None,
        slew_overhead_s: float = 180.0,
        focus_time_s: float = 300.0,
        standard_time_s: float = 300.0,
        nightly_budget_hours: float = 5.0,
        altitude_min_deg: float = 15.0,
        airmass_range: Tuple[float, float] = (1.0, 2.8),
    ):
        if location is None:
            location = EarthLocation(
                lat=33.3563 * u.deg,
                lon=-116.8648 * u.deg,
                height=1706 * u.m,
            )

        self.location = location
        self.observer = Observer(location=self.location, name="Palomar", timezone="UTC")

        self.slew_overhead_s = float(slew_overhead_s)
        self.focus_time_s = float(focus_time_s)
        self.standard_time_s = float(standard_time_s)
        self.nightly_budget_s = float(nightly_budget_hours) * 3600.0

        self.altitude_min_deg = float(altitude_min_deg)
        self.airmass_range = airmass_range

    def _night_bounds(self, base_date: Time) -> Tuple[Time, Time]:
        start_time = self.observer.twilight_evening_nautical(base_date, which="next")
        end_time = self.observer.twilight_morning_astronomical(start_time, which="next")
        return start_time, end_time

    def build_target_table(self, requests: List[FollowupRequest], current_time: Time) -> pd.DataFrame:
        rows = []
        for req in requests:
            if req.status != "PENDING":
                continue

            coord = SkyCoord(ra=req.ra * u.deg, dec=req.dec * u.deg)
            fixed_object = FixedTarget(coord=coord, name=req.candidate_id)

            obs_total = req.exptime_s + self.slew_overhead_s

            rows.append(
                {
                    "request_id": req.request_id,
                    "candidate_id": req.candidate_id,
                    "slot_idx": req.slot_idx,
                    "priority": float(req.priority),
                    "objname": req.candidate_id,
                    "ra": float(req.ra),
                    "dec": float(req.dec),
                    "fixed_object": fixed_object,
                    "SkyCoords": coord,
                    "obs_seq": {
                        "ifu_exptime": 0,
                        "rc": True,
                        "rc_obs_dict": {"obs_order": "gri", "obs_exptime": req.exptime_s},
                        "total": float(obs_total),
                    },
                    "maxairmass": self.airmass_range[1],
                    "typedesig": "f",
                }
            )

        if len(rows) == 0:
            return pd.DataFrame()

        targets = pd.DataFrame(rows)
        targets = self.update_targets_coords(targets, current_time)
        return targets

    def update_targets_coords(self, target_list: pd.DataFrame, obsdatetime: Time) -> pd.DataFrame:
        if len(target_list) == 0:
            return target_list

        out = target_list.copy()

        start_alt = []
        end_alt = []
        start_airmass = []
        end_airmass = []
        set_time = []

        for row in out.itertuples():
            coord = row.SkyCoords

            start_altaz = coord.transform_to(AltAz(obstime=obsdatetime, location=self.location))
            finish = obsdatetime + TimeDelta(row.obs_seq["total"], format="sec")
            end_altaz = coord.transform_to(AltAz(obstime=finish, location=self.location))

            try:
                target_set = self.observer.target_set_time(
                    obsdatetime,
                    row.fixed_object,
                    which="next",
                    horizon=0 * u.deg,
                )
                target_set_val = float(target_set.jd)
            except Exception:
                fallback_set = obsdatetime + TimeDelta(10 * 3600, format="sec")
                target_set_val = float(fallback_set.jd)

            start_alt.append(float(start_altaz.alt.deg))
            end_alt.append(float(end_altaz.alt.deg))

            sa = float(start_altaz.secz) if np.isfinite(start_altaz.secz) else np.nan
            ea = float(end_altaz.secz) if np.isfinite(end_altaz.secz) else np.nan
            start_airmass.append(sa)
            end_airmass.append(ea)
            set_time.append(target_set_val)

        out["start_alt"] = start_alt
        out["end_alt"] = end_alt
        out["start_airmass"] = start_airmass
        out["end_airmass"] = end_airmass
        out["set_time"] = set_time

        return out

    def get_next_observable_target(
        self,
        target_list: pd.DataFrame,
        obsdatetime: Time,
        remaining_budget_s: float,
        do_airmass: bool = True,
        do_moon_sep: bool = True,
        sort_columns: Tuple[str, ...] = ("priority", "set_time"),
        sort_order: Tuple[bool, ...] = (False, False),
        update_coords: bool = True,
    ) -> Tuple[Optional[str], Optional[dict]]:
        if len(target_list) == 0:
            return None, None

        if update_coords:
            target_list = self.update_targets_coords(target_list, obsdatetime)

        target_list = target_list.sort_values(list(sort_columns), ascending=list(sort_order))

        for row in target_list.itertuples():
            total_time_s = float(row.obs_seq["total"])
            if total_time_s > remaining_budget_s:
                continue

            start = obsdatetime
            finish = start + TimeDelta(total_time_s, format="sec")

            constraints = [AltitudeConstraint(min=self.altitude_min_deg * u.deg)]

            if do_airmass:
                constraints.append(AirmassConstraint(min=self.airmass_range[0], max=row.maxairmass))

            moon = get_body("moon", start, location=self.location)
            moon_dist = moon.separation(row.SkyCoords).deg

            if do_moon_sep:
                moon_illum = self.observer.moon_illumination(start) * 100.0
                min_moon_sep = 5.0 + max(0.0, moon_illum - 75.0)
                constraints.append(MoonSeparationConstraint(min=min_moon_sep * u.deg))

            observable = is_observable(
                constraints,
                self.observer,
                row.fixed_object,
                times=[start, finish],
                time_grid_resolution=0.1 * u.hour,
            )

            if not observable:
                continue

            s_air = float(row.start_airmass) if np.isfinite(row.start_airmass) else 99.0
            e_air = float(row.end_airmass) if np.isfinite(row.end_airmass) else 99.0

            if s_air > 3.5 or e_air > 3.5:
                continue

            chosen = {
                "request_id": row.request_id,
                "candidate_id": row.candidate_id,
                "slot_idx": row.slot_idx,
                "start_time": start,
                "finish_time": finish,
                "obs_seq": row.obs_seq,
                "moon_dist": float(moon_dist),
                "start_airmass": s_air,
                "end_airmass": e_air,
                "start_alt": float(row.start_alt),
                "end_alt": float(row.end_alt),
                "duration_s": total_time_s,
            }
            return row.request_id, chosen

        return None, None

    def simulate_night(
        self,
        requests: List[FollowupRequest],
        night_date: Time,
        do_focus: bool = True,
        do_standard: bool = True,
        sort_columns: Tuple[str, ...] = ("priority", "start_alt"),
        sort_order: Tuple[bool, ...] = (False, False),
    ) -> List[dict]:
        start_time, twilight_end = self._night_bounds(night_date)

        budget_end = start_time + TimeDelta(self.nightly_budget_s, format="sec")
        end_time = min(twilight_end, budget_end)

        targets = self.build_target_table(requests, start_time)
        if len(targets) == 0:
            return []

        current_time = start_time
        executed = []

        if do_focus:
            current_time += TimeDelta(self.focus_time_s, format="sec")
            do_focus = False

        if do_standard:
            current_time += TimeDelta(self.standard_time_s, format="sec")
            do_standard = False

        while current_time < end_time and len(targets) > 0:
            remaining_budget_s = (end_time - current_time).sec
            if remaining_budget_s <= 0:
                break

            targets = self.update_targets_coords(targets, current_time)
            targets = targets.sort_values(list(sort_columns), ascending=list(sort_order))

            req_id, chosen = self.get_next_observable_target(
                targets,
                obsdatetime=current_time,
                remaining_budget_s=remaining_budget_s,
                do_airmass=True,
                do_moon_sep=True,
                sort_columns=("priority", "set_time"),
                sort_order=(False, False),
                update_coords=False,
            )

            if req_id is None:
                current_time += TimeDelta(300, format="sec")
                continue

            executed.append(chosen)
            targets = targets[targets.request_id != req_id]
            current_time += TimeDelta(chosen["duration_s"], format="sec")

        return executed


# ============================================================
# Main environment
# ============================================================
class KilonovaSEDMEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        scenario_dirs: List[str],
        hist_sedm_csv: str,
        k_max: int,
        n_obs_max: int,
        n_days: int = 7,
        initial_days_revealed: float = 2.0,
        expire_after_days: int = 7,
        non_execution_penalty: float = 0.0,
        episode_start_time: str = "2026-01-01 00:00:00",
        min_filters_for_reward: int = 1,
        reward_filters: Tuple[str, ...] = ALL_MODEL_FILTERS,
        nightly_budget_hours: float = 5.0,
        slew_overhead_s: float = 180.0,
        seed: Optional[int] = None,
    ):
        super().__init__()

        if len(scenario_dirs) == 0:
            raise ValueError("scenario_dirs is empty")

        self.scenario_dirs = scenario_dirs
        self.hist_sedm_csv = hist_sedm_csv
        self.K_MAX = int(k_max)
        self.N_OBS_MAX = int(n_obs_max)
        self.N_DAYS = int(n_days)
        self.initial_days_revealed = float(initial_days_revealed)
        self.expire_after_days = int(expire_after_days)
        self.non_execution_penalty = float(non_execution_penalty)
        self.min_filters_for_reward = int(min_filters_for_reward)
        self.reward_filters = tuple(reward_filters)

        self.nightly_budget_hours = float(nightly_budget_hours)
        self.slew_overhead_s = float(slew_overhead_s)

        self.rng = np.random.default_rng(seed)
        self.ref_time = Time(episode_start_time, scale="utc")

        self.sedm_model = SEDMObservationModel(hist_csv=self.hist_sedm_csv)
        self.scheduler = SimplifiedSEDMScheduler(
            slew_overhead_s=self.slew_overhead_s,
            nightly_budget_hours=self.nightly_budget_hours,
        )

        self.G_scalar, self.G_bins = self._infer_g_dims()

        self.observation_space = spaces.Tuple((
            spaces.Box(-np.inf, np.inf, shape=(self.G_scalar,), dtype=np.float32),
            spaces.Box(-np.inf, np.inf, shape=(self.G_bins,), dtype=np.float32),
            spaces.Box(-np.inf, np.inf, shape=(self.K_MAX, self.N_OBS_MAX, 5), dtype=np.float32),
            spaces.MultiBinary(self.K_MAX),
            spaces.MultiBinary((self.K_MAX, self.N_OBS_MAX)),
            spaces.Box(0.0, float(self.N_DAYS), shape=(1,), dtype=np.float32),
        ))

        self.action_space = spaces.MultiDiscrete([4] * self.K_MAX)

        self.current_day = 0
        self.scenario_dir: Optional[str] = None

        self.gw_scalar_vec: Optional[np.ndarray] = None
        self.chirp_mass_bins: Optional[np.ndarray] = None

        self.candidates: Dict[str, CandidateState] = {}
        self.slot_to_source: Dict[int, str] = {}
        self.source_to_slot: Dict[str, int] = {}

        self.truth_map: Dict[str, pd.DataFrame] = {}
        self.rubin_obs_map: Dict[str, pd.DataFrame] = {}

        self.pending_requests: List[FollowupRequest] = []
        self.requested_last_step: List[FollowupRequest] = []
        self.executed_last_step: List[FollowupRequest] = []

        self._obs_tensor: Optional[np.ndarray] = None
        self._obs_mask: Optional[np.ndarray] = None
        self._cand_mask: Optional[np.ndarray] = None

        self.last_rubin_reveal_time: float = 0.0
        self._request_counter: int = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        self.current_day = 0
        self.pending_requests = []
        self.requested_last_step = []
        self.executed_last_step = []
        self._request_counter = 0

        self._load_random_scenario()
        self._initialize_buffers()

        self.last_rubin_reveal_time = float(self.initial_days_revealed)
        self._reveal_rubin_data_until(self.last_rubin_reveal_time)

        obs = self._build_observation()
        info = self._build_info(initial=True)
        return obs, info

    def step(self, action):
        action = np.asarray(action, dtype=int)
        if action.shape != (self.K_MAX,):
            raise ValueError(f"Expected action shape {(self.K_MAX,)}, got {action.shape}")

        before_fit_maps = self._compute_followup_filter_fit_maps(action)
        self.requested_last_step = self._enqueue_requests_from_action(action)

        night_date = self.ref_time + TimeDelta(self.current_day * 86400, format="sec")
        self.executed_last_step = self._run_scheduler_one_night(night_date)
        self._materialize_executed_requests(self.executed_last_step)

        self.current_day += 1

        new_t_hi = self.initial_days_revealed + self.current_day
        self._reveal_rubin_data_between(self.last_rubin_reveal_time, new_t_hi)
        self.last_rubin_reveal_time = new_t_hi

        after_fit_maps = self._compute_followup_filter_fit_maps_from_requests(self.requested_last_step)

        reward, reward_breakdown = self._compute_reward(
            requested=self.requested_last_step,
            executed=self.executed_last_step,
            before_fit_maps=before_fit_maps,
            after_fit_maps=after_fit_maps,
        )

        obs = self._build_observation()
        terminated = self._termination_condition()
        truncated = bool(self.current_day >= self.N_DAYS and not terminated)

        info = self._build_info(
            initial=False,
            reward_breakdown=reward_breakdown,
            requested=self.requested_last_step,
            executed=self.executed_last_step,
        )
        return obs, reward, terminated, truncated, info

    def render(self):
        print(f"Day: {self.current_day}")
        print(f"Scenario: {self.scenario_dir}")
        print(f"Candidates: {len(self.candidates)}")
        print(f"Pending requests: {sum(r.status == 'PENDING' for r in self.pending_requests)}")

    def _infer_g_dims(self) -> Tuple[int, int]:
        meta_path = os.path.join(self.scenario_dirs[0], "scenario_metadata.csv")
        meta = safe_read_csv(meta_path)
        row = meta.iloc[0]

        g_scalar = len(build_gw_scalar_vector(row))
        g_bins = len(build_chirp_mass_bins(row))
        return g_scalar, g_bins

    def _load_random_scenario(self) -> None:
        self.scenario_dir = str(self.rng.choice(self.scenario_dirs))
        self._load_scenario(self.scenario_dir)

    def _load_scenario(self, scenario_dir: str) -> None:
        meta = safe_read_csv(os.path.join(scenario_dir, "scenario_metadata.csv"))
        true_all = safe_read_csv(os.path.join(scenario_dir, "true_all.csv"))
        obs_all = safe_read_csv(os.path.join(scenario_dir, "observe_all.csv"))

        row = meta.iloc[0]
        self.gw_scalar_vec = build_gw_scalar_vector(row).astype(np.float32)
        self.chirp_mass_bins = build_chirp_mass_bins(row).astype(np.float32)

        for df in [true_all, obs_all]:
            df["source_id"] = df["source_id"].astype(str)
            df["filter"] = df["filter"].astype(str).str.strip().str.lower()
            df["time"] = pd.to_numeric(df["time"], errors="coerce")
            df["ra"] = pd.to_numeric(df["ra"], errors="coerce")
            df["dec"] = pd.to_numeric(df["dec"], errors="coerce")
            df["z"] = pd.to_numeric(df["z"], errors="coerce")
            if "mag" in df.columns:
                df["mag"] = pd.to_numeric(df["mag"], errors="coerce")
            if "magerr" in df.columns:
                df["magerr"] = pd.to_numeric(df["magerr"], errors="coerce")
            if "limmag" in df.columns:
                df["limmag"] = pd.to_numeric(df["limmag"], errors="coerce")

        source_ids = sorted(true_all["source_id"].unique().tolist())
        if len(source_ids) > self.K_MAX:
            raise ValueError(f"Scenario has {len(source_ids)} candidates but K_MAX={self.K_MAX}")

        self.candidates = {}
        self.slot_to_source = {}
        self.source_to_slot = {}

        for slot_idx, sid in enumerate(source_ids):
            rows = true_all[true_all["source_id"] == sid]
            first = rows.iloc[0]
            cand_type = str(first.get("type", "unknown"))
            is_kn = cand_type.upper() == "KN"

            cand = CandidateState(
                source_id=sid,
                slot_idx=slot_idx,
                cand_type=cand_type,
                ra=float(first["ra"]),
                dec=float(first["dec"]),
                z=float(first["z"]),
                is_kn=is_kn,
            )
            self.candidates[sid] = cand
            self.slot_to_source[slot_idx] = sid
            self.source_to_slot[sid] = slot_idx

        self.truth_map = {
            sid: g.copy().sort_values(["time", "filter"]).reset_index(drop=True)
            for sid, g in true_all.groupby("source_id")
        }
        self.rubin_obs_map = {
            sid: g.copy().sort_values(["time", "filter"]).reset_index(drop=True)
            for sid, g in obs_all.groupby("source_id")
        }

    def _initialize_buffers(self) -> None:
        self._obs_tensor = np.full((self.K_MAX, self.N_OBS_MAX, 5), np.nan, dtype=np.float32)
        self._obs_mask = np.zeros((self.K_MAX, self.N_OBS_MAX), dtype=np.int8)
        self._cand_mask = np.zeros((self.K_MAX,), dtype=np.int8)

        for slot_idx in self.slot_to_source:
            self._cand_mask[slot_idx] = 1

        for cand in self.candidates.values():
            cand.observed_rows = []
            cand.mag_fit_before = {}
            cand.mag_fit_after = {}

    def _build_observation(self):
        return (
            self.gw_scalar_vec.astype(np.float32),
            self.chirp_mass_bins.astype(np.float32),
            self._obs_tensor.astype(np.float32),
            self._cand_mask.astype(np.int8),
            self._obs_mask.astype(np.int8),
            np.asarray([self.current_day], dtype=np.float32),
        )

    def _append_observation_row(self, slot_idx: int, row: dict) -> None:
        free = np.where(self._obs_mask[slot_idx] == 0)[0]
        if len(free) == 0:
            self._obs_tensor[slot_idx, :-1] = self._obs_tensor[slot_idx, 1:]
            self._obs_mask[slot_idx, :-1] = self._obs_mask[slot_idx, 1:]
            j = self.N_OBS_MAX - 1
        else:
            j = int(free[0])

        filt = str(row["filter"]).lower()
        if filt not in FILTER_TO_ID:
            return

        arr = np.array([
            float(row.get("time", np.nan)),
            float(row.get("mag", np.nan)),
            float(row.get("magerr", np.nan)),
            float(row.get("limmag", np.nan)),
            float(FILTER_TO_ID[filt]),
        ], dtype=np.float32)

        self._obs_tensor[slot_idx, j] = arr
        self._obs_mask[slot_idx, j] = 1

    def _reveal_rubin_data_until(self, t_max: float) -> None:
        for sid, df in self.rubin_obs_map.items():
            rows = df[df["time"] <= t_max].to_dict("records")
            self._insert_new_rows(sid, rows, origin="Rubin")

    def _reveal_rubin_data_between(self, t_lo: float, t_hi: float) -> None:
        for sid, df in self.rubin_obs_map.items():
            rows = df[(df["time"] > t_lo) & (df["time"] <= t_hi)].to_dict("records")
            self._insert_new_rows(sid, rows, origin="Rubin")

    def _insert_new_rows(self, source_id: str, rows: List[dict], origin: str) -> None:
        if source_id not in self.candidates or len(rows) == 0:
            return

        cand = self.candidates[source_id]
        slot_idx = cand.slot_idx

        formatted = []
        for r in rows:
            rr = {
                "time": float(r.get("time", np.nan)),
                "filter": str(r.get("filter", "")).lower(),
                "mag": float(r.get("mag", np.nan)),
                "magerr": float(r.get("magerr", np.nan)) if "magerr" in r else np.nan,
                "limmag": float(r.get("limmag", np.nan)) if "limmag" in r else np.nan,
                "origin": origin,
            }
            formatted.append(rr)
            self._append_observation_row(slot_idx, rr)

        cand.add_rows(formatted)

    def _candidate_has_pending_request(self, sid: str) -> bool:
        return any(r.candidate_id == sid and r.status == "PENDING" for r in self.pending_requests)

    def _compute_followup_filter_fit_maps(self, action: np.ndarray) -> Dict[str, Dict[str, float]]:
        out: Dict[str, Dict[str, float]] = {}
        for slot_idx, a in enumerate(action):
            sid = self.slot_to_source.get(slot_idx)
            if sid is None or int(a) == 0:
                continue
            out[sid] = compute_mag_fit_chi2_by_filter(
                self.candidates[sid].observed_df(),
                allowed_filters=self.reward_filters,
                min_points_per_filter=2,
            )
        return out

    def _compute_followup_filter_fit_maps_from_requests(
        self,
        requests: List[FollowupRequest],
    ) -> Dict[str, Dict[str, float]]:
        out: Dict[str, Dict[str, float]] = {}
        seen = set()
        for req in requests:
            sid = req.candidate_id
            if sid in seen:
                continue
            seen.add(sid)
            out[sid] = compute_mag_fit_chi2_by_filter(
                self.candidates[sid].observed_df(),
                allowed_filters=self.reward_filters,
                min_points_per_filter=2,
            )
        return out

    def _enqueue_requests_from_action(self, action: np.ndarray) -> List[FollowupRequest]:
        requests = []
        for slot_idx, a in enumerate(action):
            a = int(a)
            sid = self.slot_to_source.get(slot_idx)
            if sid is None or a == 0:
                continue

            if self._candidate_has_pending_request(sid):
                continue

            exptime = ACTION_TO_EXPTIME[a]
            cand = self.candidates[sid]

            current_fit_map = compute_mag_fit_chi2_by_filter(
                cand.observed_df(),
                allowed_filters=self.reward_filters,
                min_points_per_filter=2,
            )

            if len(current_fit_map) > 0:
                mean_chi2 = float(np.mean(list(current_fit_map.values())))
            else:
                mean_chi2 = 10.0

            priority = 1.0 / (1.0 + mean_chi2)

            req = FollowupRequest(
                request_id=f"req_{self.current_day}_{self._request_counter}_{sid}",
                candidate_id=sid,
                slot_idx=slot_idx,
                exptime_s=int(exptime),
                request_day=self.current_day,
                priority=float(priority),
                ra=cand.ra,
                dec=cand.dec,
            )
            self._request_counter += 1

            self.pending_requests.append(req)
            requests.append(req)

        return requests

    def _run_scheduler_one_night(self, night_date: Time) -> List[FollowupRequest]:
        for req in self.pending_requests:
            if req.status == "PENDING" and (self.current_day - req.request_day) > self.expire_after_days:
                req.status = "EXPIRED"
                req.reason = "expired"

        pending = [r for r in self.pending_requests if r.status == "PENDING"]
        if len(pending) == 0:
            return []

        executed_sched = self.scheduler.simulate_night(
            requests=pending,
            night_date=night_date,
            do_focus=True,
            do_standard=True,
            sort_columns=("priority", "start_alt"),
            sort_order=(False, False),
        )

        executed_by_request_id = {e["request_id"]: e for e in executed_sched}
        executed_requests = []

        for req in pending:
            if req.request_id in executed_by_request_id:
                entry = executed_by_request_id[req.request_id]
                abs_start = entry["start_time"]
                rel_days = (abs_start - self.ref_time).sec / 86400.0

                req.status = "EXECUTED"
                req.executed_day = self.current_day
                req.scheduled_time = float(rel_days)
                req.reason = "scheduled"
                executed_requests.append(req)

        return executed_requests

    def _materialize_executed_requests(self, executed_requests: List[FollowupRequest]) -> None:
        for req in executed_requests:
            sid = req.candidate_id
            truth_df = self.truth_map[sid]

            sedm_df = self.sedm_model.generate(
                source_id=sid,
                truth_df=truth_df,
                exptime_s=req.exptime_s,
                t_start=float(req.scheduled_time),
                rng=self.rng,
                max_visits=1,
            )

            rows = []
            for _, r in sedm_df.iterrows():
                rows.append({
                    "time": float(r["time"]),
                    "filter": str(r["filter"]).lower(),
                    "mag": float(r["mag"]) if np.isfinite(r["mag"]) else np.nan,
                    "magerr": float(r["magerr"]) if np.isfinite(r["magerr"]) else np.nan,
                    "limmag": float(r["limiting_mag"]) if np.isfinite(r["limiting_mag"]) else np.nan,
                    "origin": "SEDM",
                })

            self._insert_new_rows(sid, rows, origin="SEDM")

    def _compute_reward(
        self,
        requested: List[FollowupRequest],
        executed: List[FollowupRequest],
        before_fit_maps: Dict[str, Dict[str, float]],
        after_fit_maps: Dict[str, Dict[str, float]],
    ) -> Tuple[float, Dict[str, Any]]:
        reward = 0.0
        executed_ids = set(r.candidate_id for r in executed)
        per_candidate = {}
        seen = set()

        for req in requested:
            sid = req.candidate_id
            if sid in seen:
                continue
            seen.add(sid)

            before_map = before_fit_maps.get(sid, {})
            after_map = after_fit_maps.get(sid, {})

            avg_improvement, valid_filters, per_filter_improvement = compute_average_filter_improvement(
                before_map=before_map,
                after_map=after_map,
                min_filters_required=self.min_filters_for_reward,
            )

            contrib = float(avg_improvement)
            if sid not in executed_ids:
                contrib -= self.non_execution_penalty

            reward += contrib
            per_candidate[sid] = {
                "fit_before": before_map,
                "fit_after": after_map,
                "valid_filters_used": valid_filters,
                "n_valid_filters": len(valid_filters),
                "avg_improvement": avg_improvement,
                "per_filter_improvement": per_filter_improvement,
                "executed": sid in executed_ids,
                "reward_contribution": contrib,
            }

            self.candidates[sid].mag_fit_before = before_map
            self.candidates[sid].mag_fit_after = after_map

        return float(reward), {
            "requested_ids": [r.candidate_id for r in requested],
            "executed_ids": list(executed_ids),
            "per_candidate": per_candidate,
        }

    def _termination_condition(self) -> bool:
        scores = []
        for sid, cand in self.candidates.items():
            fit_map = compute_mag_fit_chi2_by_filter(
                cand.observed_df(),
                allowed_filters=self.reward_filters,
                min_points_per_filter=2,
            )
            if len(fit_map) >= self.min_filters_for_reward:
                mean_chi2 = float(np.mean(list(fit_map.values())))
                scores.append((sid, mean_chi2, cand.is_kn))

        if len(scores) == 0:
            return False

        scores = sorted(scores, key=lambda x: x[1])
        best_sid, best_mean_chi2, best_is_kn = scores[0]

        if not best_is_kn:
            return False

        if len(scores) == 1:
            return True

        second_mean_chi2 = scores[1][1]
        return (second_mean_chi2 - best_mean_chi2) > 0.5

    def _build_info(
        self,
        initial: bool = False,
        reward_breakdown: Optional[dict] = None,
        requested: Optional[List[FollowupRequest]] = None,
        executed: Optional[List[FollowupRequest]] = None,
    ) -> Dict[str, Any]:

        info = {
            "current_day": self.current_day,
            "n_candidates": len(self.candidates),
            "n_pending_requests": sum(r.status == "PENDING" for r in self.pending_requests),
        }

        if initial:
            return info

        requested = requested or []
        executed = executed or []

        info["requested"] = [
            {
                "candidate_id": r.candidate_id,
                "exptime_s": r.exptime_s,
                "status": r.status,
            }
            for r in requested
        ]

        executed_summary = []
        for r in executed:
            sid = r.candidate_id
            cand_df = self.candidates[sid].observed_df()

            sedm_df = cand_df[cand_df["origin"] == "SEDM"].copy()
            sedm_df["mag"] = pd.to_numeric(sedm_df["mag"], errors="coerce")

            det_df = sedm_df[np.isfinite(sedm_df["mag"])].copy()

            executed_summary.append(
                {
                    "candidate_id": sid,
                    "exptime_s": r.exptime_s,
                    "scheduled_time": r.scheduled_time,
                    "n_sedm_points": int(len(sedm_df)),
                    "n_sedm_detections": int(len(det_df)),
                    "detected_filters": sorted(det_df["filter"].astype(str).unique().tolist()),
                    "has_sedm_detection": bool(len(det_df) > 0),
                }
            )

        info["executed"] = executed_summary
        info["reward"] = reward_breakdown or {}

        return info


# ============================================================
# Exact KN matching helpers
# ============================================================
def _is_exact_kn_sid(sid: str, target_id: str) -> bool:
    sid_str = str(sid).strip()
    return sid_str in {f"KN_{target_id}", target_id}


def find_scenario_with_target(scenario_dirs: List[str], target_id: str = "8026") -> List[Tuple[str, pd.DataFrame]]:
    matches = []

    for sdir in scenario_dirs:
        true_all_path = os.path.join(sdir, "true_all.csv")
        if not os.path.exists(true_all_path):
            continue

        df = pd.read_csv(true_all_path, usecols=["source_id", "type"])
        df["source_id"] = df["source_id"].astype(str).str.strip()
        df["type"] = df["type"].astype(str).str.strip()

        mask = (df["type"].str.upper() == "KN") & df["source_id"].apply(lambda x: _is_exact_kn_sid(x, target_id))

        if mask.any():
            matches.append((sdir, df.loc[mask].copy()))

    return matches


# ============================================================
# Example usage
# ============================================================
if __name__ == "__main__":
    SCENARIO_ROOT = "/projects/bdws/npletskova1/git/Pythia/pythia/train_data_gen/BNS_O5a_combined"
    HIST_SEDM_CSV = "/projects/bdws/npletskova1/git/Pythia/pythia/train_data_gen/ZTF/all_sedm_phot_20260220_with_skynoise.csv"
    TARGET_KN_ID = "8026"

    scenario_dirs = sorted(glob.glob(os.path.join(SCENARIO_ROOT, "scenario_*")))
    print(f"Found {len(scenario_dirs)} scenarios")

    matches = find_scenario_with_target(scenario_dirs, TARGET_KN_ID)

    if len(matches) == 0:
        raise RuntimeError(f"Exact KN_{TARGET_KN_ID} was not found in any scenario.")

    print(f"\nFound exact KN_{TARGET_KN_ID} in {len(matches)} scenario(s):")
    for sdir, rows in matches:
        print(f"  {sdir}")
        print(rows[["source_id", "type"]].drop_duplicates().to_string(index=False))

    forced_scenario_dir = matches[0][0]
    print(f"\nUsing scenario: {forced_scenario_dir}")

    env = KilonovaSEDMEnv(
        scenario_dirs=[forced_scenario_dir],
        hist_sedm_csv=HIST_SEDM_CSV,
        k_max=7335,
        n_obs_max=128,
        n_days=7,
        initial_days_revealed=1.0,
        expire_after_days=6,
        non_execution_penalty=0,
        episode_start_time="2026-01-01 00:00:00",
        min_filters_for_reward=1,
        reward_filters=("g", "r", "i", "u", "z", "y"),
        nightly_budget_hours=5.0,
        slew_overhead_s=180.0,
        seed=22,
    )

    obs, info = env.reset()
    print("Reset info:", info)
    print("Loaded scenario:", env.scenario_dir)

    chosen_kn_idxs = []
    for idx, sid in env.slot_to_source.items():
        cand = env.candidates[sid]
        if cand.is_kn and _is_exact_kn_sid(sid, TARGET_KN_ID):
            chosen_kn_idxs.append(idx)

    if len(chosen_kn_idxs) == 0:
        print("\nAvailable source_ids in this scenario:")
        for idx, sid in list(env.slot_to_source.items())[:100]:
            cand = env.candidates[sid]
            print(f"  slot={idx}, source_id={sid}, type={cand.cand_type}, is_kn={cand.is_kn}")
        raise RuntimeError(f"Exact KN_{TARGET_KN_ID} not found in loaded scenario.")

    print(f"\nFollowing KN(s) from this scenario:")
    for idx in chosen_kn_idxs:
        sid = env.slot_to_source[idx]
        cand = env.candidates[sid]
        print(
            f"  slot={idx}, candidate_id={sid}, "
            f"type={cand.cand_type}, is_kn={cand.is_kn}"
        )

    OTHER_CANDIDATES_TO_FOLLOW = 10

    other_idxs = []
    for idx, sid in env.slot_to_source.items():
        if idx in chosen_kn_idxs:
            continue
        other_idxs.append(idx)

    if len(other_idxs) < OTHER_CANDIDATES_TO_FOLLOW:
        print(
            f"\nWarning: only {len(other_idxs)} other candidates available, "
            f"using all of them."
        )
        OTHER_CANDIDATES_TO_FOLLOW = len(other_idxs)

    chosen_other_idxs = env.rng.choice(
        other_idxs,
        size=OTHER_CANDIDATES_TO_FOLLOW,
        replace=False
    ).tolist() if OTHER_CANDIDATES_TO_FOLLOW > 0 else []

    tracked_idxs = chosen_kn_idxs + chosen_other_idxs
    tracked_sids = [env.slot_to_source[idx] for idx in tracked_idxs]

    print("\nTracking exact KN + 10 other candidates from this scenario:")
    for idx in tracked_idxs:
        sid = env.slot_to_source[idx]
        cand = env.candidates[sid]
        print(
            f"  slot={idx}, candidate_id={sid}, "
            f"type={cand.cand_type}, is_kn={cand.is_kn}"
        )

    for day in range(env.N_DAYS):
        action = np.zeros(env.K_MAX, dtype=int)

        for idx in tracked_idxs:
            if idx in chosen_kn_idxs:
                action[idx] = 3
            else:
                action[idx] = 2

        obs, reward, terminated, truncated, info = env.step(action)

        print(f"\n================ Day {day} ================")
        print("Reward:", reward)
        print("Current day:", info["current_day"])
        print("Requested:", info["requested"])
        print("Executed:", info["executed"])

        if len(info["executed"]) > 0:
            print("Executed candidate details:")
            for ex in info["executed"]:
                print(
                    f"  {ex['candidate_id']}: "
                    f"exptime={ex['exptime_s']} s, "
                    f"scheduled_time={ex['scheduled_time']:.4f}, "
                    f"n_sedm_points={ex['n_sedm_points']}, "
                    f"n_sedm_detections={ex['n_sedm_detections']}, "
                    f"detected_filters={ex['detected_filters']}, "
                    f"has_sedm_detection={ex['has_sedm_detection']}"
                )

        print("\nTracked candidate history:")

        for idx in tracked_idxs:
            sid = env.slot_to_source[idx]
            cand = env.candidates[sid]
            df = cand.observed_df().copy()

            rubin_df = df[df["origin"] == "Rubin"].copy()
            sedm_df = df[df["origin"] == "SEDM"].copy()

            rubin_df["mag"] = pd.to_numeric(rubin_df["mag"], errors="coerce")
            sedm_df["mag"] = pd.to_numeric(sedm_df["mag"], errors="coerce")

            n_rubin_rows = len(rubin_df)
            n_sedm_rows = len(sedm_df)

            n_rubin_det = int(np.isfinite(rubin_df["mag"]).sum()) if n_rubin_rows > 0 else 0
            n_sedm_det = int(np.isfinite(sedm_df["mag"]).sum()) if n_sedm_rows > 0 else 0

            print(
                f"  {sid} ({cand.cand_type}): "
                f"Rubin rows={n_rubin_rows}, Rubin detections={n_rubin_det}, "
                f"SEDM rows={n_sedm_rows}, SEDM detections={n_sedm_det}"
            )

        reward_pc = info.get("reward", {}).get("per_candidate", {})

        print("\nPer-candidate reward contribution:")
        for idx in tracked_idxs:
            sid = env.slot_to_source[idx]
            cand = env.candidates[sid]

            if sid in reward_pc:
                rc = reward_pc[sid]
                print(
                    f"  {sid} ({cand.cand_type}): "
                    f"executed={rc['executed']}, "
                    f"avg_improvement={rc['avg_improvement']:.4f}, "
                    f"n_valid_filters={rc['n_valid_filters']}, "
                    f"reward_contribution={rc['reward_contribution']:.4f}"
                )
            else:
                print(f"  {sid} ({cand.cand_type}): no reward entry this step")

        executed_ids = {ex["candidate_id"] for ex in info["executed"]}
        n_exec_tracked = sum(1 for sid in tracked_sids if sid in executed_ids)
        print(f"\nTracked executed tonight: {n_exec_tracked}/{len(tracked_sids)}")

        print(f"\nKN_{TARGET_KN_ID} status:")
        for idx in chosen_kn_idxs:
            sid = env.slot_to_source[idx]
            cand = env.candidates[sid]
            df = cand.observed_df().copy()

            rubin_df = df[df["origin"] == "Rubin"].copy()
            sedm_df = df[df["origin"] == "SEDM"].copy()

            rubin_df["mag"] = pd.to_numeric(rubin_df["mag"], errors="coerce")
            sedm_df["mag"] = pd.to_numeric(sedm_df["mag"], errors="coerce")

            n_total = len(df)
            n_rubin = len(rubin_df)
            n_sedm = len(sedm_df)
            n_sedm_det = int(np.isfinite(sedm_df["mag"]).sum()) if n_sedm > 0 else 0

            print(
                f"  {sid}: total_obs={n_total}, "
                f"Rubin_obs={n_rubin}, "
                f"SEDM_obs={n_sedm}, "
                f"SEDM_detections={n_sedm_det}"
            )

        if terminated or truncated:
            print("\nEpisode ended.")
            break

Found 161 scenarios

Found exact KN_8026 in 1 scenario(s):
  /projects/bdws/npletskova1/git/Pythia/pythia/train_data_gen/BNS_O5a_combined/scenario_4174
source_id type
  KN_8026   KN

Using scenario: /projects/bdws/npletskova1/git/Pythia/pythia/train_data_gen/BNS_O5a_combined/scenario_4174
Reset info: {'current_day': 0, 'n_candidates': 5664, 'n_pending_requests': 0}
Loaded scenario: /projects/bdws/npletskova1/git/Pythia/pythia/train_data_gen/BNS_O5a_combined/scenario_4174

Following KN(s) from this scenario:
  slot=3491, candidate_id=KN_8026, type=KN, is_kn=True

Tracking exact KN + 10 other candidates from this scenario:
  slot=3491, candidate_id=KN_8026, type=KN, is_kn=True
  slot=1499, candidate_id=Ia_132, type=Ia, is_kn=False
  slot=3698, candidate_id=SLSN-II_4585, type=SLSN-II, is_kn=False
  slot=1127, candidate_id=IIn_1422, type=IIn, is_kn=False
  slot=4361, candidate_id=SLSN-II_5248, type=SLSN-II, is_kn=False
  slot=5343, candidate_id=SLSN-I_4058, type=SLSN-I, is_kn=False
  slot=

 (5173770.99345595, 1333468.1552454 , 3474821.26716067)] m, obsgeovel=[(-88.96776767, 378.67007376, 0.213723  ),
 (-97.22970948, 376.63332646, 0.23474691)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5143368.56392961, 1446130.09601298, 3474894.8204914 )] m, obsgeovel=[( -97.22970948, 376.63332646, 0.23474691),
 (-105.44512148, 374.41633926, 0.25565851)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5136992.73341934, 1468581.16370829, 3474910.28299867)] m, obsgeovel=[( -97.22970948, 376.63332646, 0.23474691),
 (-107.08227392, 373.95140488, 0.25982647)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5103643.57053592, 1580406.26264557, 3474991.33846544)] m, obsgeovel=[(-107.08227392, 373.95140488, 0.25982647),
 (-115.23666264, 371.51953778, 0.28059009)] m / s)>. Angular separation can depen


================ Day 0 ================
Reward: 0.0
Current day: 1
Requested: [{'candidate_id': 'II_1092', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'IIb_2682', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'IIn_1422', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'Ia_132', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'Ia_648', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'Ic-BL_3073', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'KN_8026', 'exptime_s': 180, 'status': 'EXECUTED'}, {'candidate_id': 'SLSN-II_4585', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'status': 'EXECUTED'}, {'candidate_id': 'SLSN-II_5248', 'exptime_s': 120, 'status': 'PENDING'}, {'candidate_id': 'SLSN-I_4058', 'exptime_s': 120, 'status': 'PENDING'}]
Executed: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'scheduled_time': 0.08590926571438712, 'n_sedm_points': 3, 'n_sedm_detections': 0, 'detecte

 (5145739.48925364, 1437692.69816752, 3474885.79191652)] m, obsgeovel=[( -96.61084589, 376.79254582, 0.23328287),
 (-104.82988883, 374.58905795, 0.2542084 )] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5113063.89645104, 1549716.47058921, 3474965.16588084)] m, obsgeovel=[(-104.82988883, 374.58905795, 0.2542084 ),
 (-112.99876485, 372.20630845, 0.27501232)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5106235.13902466, 1572033.94578179, 3474981.78855535)] m, obsgeovel=[(-104.82988883, 374.58905795, 0.2542084 ),
 (-114.62617557, 371.708346  , 0.27915764)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (5070630.34711359, 1683161.42868819, 3475068.62384023)] m, obsgeovel=[(-114.62617557, 371.708346  , 0.27915764),
 (-122.72969332, 369.11199524, 0.29980249)] m / s)>. Angular separation can dep


================ Day 1 ================
Reward: 0.0
Current day: 2
Requested: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'status': 'EXECUTED'}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'status': 'EXECUTED'}]
Executed: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'scheduled_time': 1.0863904037202399, 'n_sedm_points': 6, 'n_sedm_detections': 3, 'detected_filters': ['g', 'i', 'r'], 'has_sedm_detection': True}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'scheduled_time': 1.0829181814980176, 'n_sedm_points': 6, 'n_sedm_detections': 0, 'detected_filters': [], 'has_sedm_detection': False}]
Executed candidate details:
  KN_8026: exptime=180 s, scheduled_time=1.0864, n_sedm_points=6, n_sedm_detections=3, detected_filters=['g', 'i', 'r'], has_sedm_detection=True
  SLSN-II_4620: exptime=120 s, scheduled_time=1.0829, n_sedm_points=6, n_sedm_detections=0, detected_filters=[], has_sedm_detection=False

Tracked candidate history:
  KN_8026 (KN): Rubin rows=0, Rubin detections

...


================ Day 4 ================
Reward: 1.6306346237635314
Current day: 5
Requested: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'status': 'EXECUTED'}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'status': 'EXECUTED'}]
Executed: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'scheduled_time': 4.0878925762760145, 'n_sedm_points': 15, 'n_sedm_detections': 9, 'detected_filters': ['g', 'i', 'r'], 'has_sedm_detection': True}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'scheduled_time': 4.084420354053792, 'n_sedm_points': 15, 'n_sedm_detections': 0, 'detected_filters': [], 'has_sedm_detection': False}]
Executed candidate details:
  KN_8026: exptime=180 s, scheduled_time=4.0879, n_sedm_points=15, n_sedm_detections=9, detected_filters=['g', 'i', 'r'], has_sedm_detection=True
  SLSN-II_4620: exptime=120 s, scheduled_time=4.0844, n_sedm_points=15, n_sedm_detections=0, detected_filters=[], has_sedm_detection=False

Tracked candidate history:
  KN_8026 (KN): Rubin rows=0

 (5011583.45563942, 1851334.97773876, 3475200.21291638)] m, obsgeovel=[(-126.98058862, 367.67126701, 0.31060611),
 (-134.99284089, 364.80563353, 0.33104816)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4969891.79716081, 1960324.98562162, 3475302.56636067)] m, obsgeovel=[(-134.99284089, 364.80563353, 0.33104816),
 (-142.94049161, 361.76542035, 0.3513318 )] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4961267.91145195, 1982012.08009271, 3475323.76587499)] m, obsgeovel=[(-134.99284089, 364.80563353, 0.33104816),
 (-144.52193431, 361.13655472, 0.35536867)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4916729.79236215, 2089870.18401305, 3475433.38525767)] m, obsgeovel=[(-144.52193431, 361.13655472, 0.35536867),
 (-152.38704554, 357.88877368, 0.37544942)] m / s)>. Angular separation can dep


================ Day 5 ================
Reward: 0.0
Current day: 6
Requested: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'status': 'EXECUTED'}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'status': 'EXECUTED'}]
Executed: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'scheduled_time': 5.088411640065412, 'n_sedm_points': 18, 'n_sedm_detections': 10, 'detected_filters': ['g', 'i', 'r'], 'has_sedm_detection': True}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'scheduled_time': 5.08493941784319, 'n_sedm_points': 18, 'n_sedm_detections': 0, 'detected_filters': [], 'has_sedm_detection': False}]
Executed candidate details:
  KN_8026: exptime=180 s, scheduled_time=5.0884, n_sedm_points=18, n_sedm_detections=10, detected_filters=['g', 'i', 'r'], has_sedm_detection=True
  SLSN-II_4620: exptime=120 s, scheduled_time=5.0849, n_sedm_points=18, n_sedm_detections=0, detected_filters=[], has_sedm_detection=False

Tracked candidate history:
  KN_8026 (KN): Rubin rows=2, Rubin detecti

 (4972534.69466804, 1953626.1663368 , 3475294.22711497)] m, obsgeovel=[(-134.50018908, 364.98755624, 0.32972575),
 (-142.45193714, 361.95807614, 0.35002247)] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4928614.82826541, 2061737.49835932, 3475402.25225127)] m, obsgeovel=[(-142.45193714, 361.95807614, 0.35002247),
 (-150.33551405, 358.75537904, 0.3701517 )] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4919547.63746537, 2083243.02006768, 3475424.58047072)] m, obsgeovel=[(-142.45193714, 361.95807614, 0.35002247),
 (-151.9037163 , 358.094187  , 0.3741566 )] m / s)>. Angular separation can depend on the direction of the transformation. [astropy.coordinates.baseframe]
 (4872805.14211443, 2190164.26338609, 3475539.81402336)] m, obsgeovel=[(-151.9037163 , 358.094187  , 0.3741566 ),
 (-159.70051084, 354.68565988, 0.39407214)] m / s)>. Angular separation can dep


================ Day 6 ================
Reward: 0.0
Current day: 7
Requested: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'status': 'EXECUTED'}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'status': 'EXECUTED'}]
Executed: [{'candidate_id': 'KN_8026', 'exptime_s': 180, 'scheduled_time': 6.088939240357528, 'n_sedm_points': 21, 'n_sedm_detections': 10, 'detected_filters': ['g', 'i', 'r'], 'has_sedm_detection': True}, {'candidate_id': 'SLSN-II_4620', 'exptime_s': 120, 'scheduled_time': 6.085467018135306, 'n_sedm_points': 21, 'n_sedm_detections': 0, 'detected_filters': [], 'has_sedm_detection': False}]
Executed candidate details:
  KN_8026: exptime=180 s, scheduled_time=6.0889, n_sedm_points=21, n_sedm_detections=10, detected_filters=['g', 'i', 'r'], has_sedm_detection=True
  SLSN-II_4620: exptime=120 s, scheduled_time=6.0855, n_sedm_points=21, n_sedm_detections=0, detected_filters=[], has_sedm_detection=False

Tracked candidate history:
  KN_8026 (KN): Rubin rows=2, Rubin detect